# TropiLite-IE: Protocol A — Electricity-Only Chronological Forecasting Notebook

This notebook implements the revised reviewer-response experiment protocol.

Main changes (Protocol A final revision):
- full GEPIII electricity subset (`meter == 0`) as primary dataset;
- chronological per-building 70/15/15 split;
- leakage-safe lag and rolling feature construction;
- verified Köppen–Geiger climate table;
- LightGBM ablation experiments;
- genuine GRU sequence forecasting with 24/48/168 hour windows;
- validation-only selection and fusion gating;
- block-bootstrap and building-level statistical evaluation.

## Reproducibility and methodological notes

1. Primary data: electricity-only GEPIII subset (`meter == 0`). Avoid arbitrary 100k pilot sampling unless compute constraints require it.

2. Split strategy:
- per-building chronological split;
- train: earliest 70%;
- validation: next 15%;
- test: latest 15%.

3. Feature leakage control:
- lags and rolling statistics are generated only from observations before the forecast origin;
- preprocessing is fitted on training partitions only.

4. Climate analysis:
- site climate labels must come from a pinned Köppen–Geiger source;
- tropical classes are Af/Am/Aw.

5. GRU:
- real sequences are used instead of single-timestep reshaping.

6. Model selection:
- validation data only;
- fusion retained only when the preregistered improvement gate is achieved.

In [ ]:
# ---------------------------------------------------------------------------
# BUILD IDENTIFIER. If this banner is not the FIRST output of the notebook,
# you are running a stale copy.
# Build corrected-2026-09-19c-generalization: adds (a) three seasonal-naive
# persistence baselines (lag-1/24/168, evaluated on the identical test rows with
# the identical metrics, no model fit) and (b) XGBoost + Lag Features and
# CatBoost + Lag Features tree-family ablations (same lag matrix, other booster).
# Superset of corrected-2026-09-19a: same seeds/gates/protocol, vector (PDF)
# figure exports, no stale alpha reference line. Previously reported models are
# bit-for-bit unchanged; new rows only ADD to the results tables.
# NOTE: with low_memory_mode ON (the Kaggle default) the printed config shows
# max_eval_windows_per_building=400 and rf_n_estimators=150. That is the
# EXPECTED low-memory cap of THIS build, NOT a sign of a stale file.
TROPILITE_BUILD = "corrected-2026-09-19c-generalization"
print("=" * 70)
print("TropiLite-IE Protocol A --", TROPILITE_BUILD)
print("low_memory_mode caps the printed config to 400/150 -- expected in this build.")
print("=" * 70)


In [ ]:
# Optional one-time installation inside the selected VS Code/Jupyter environment.
# Uncomment and run, or install from requirements.txt supplied with this notebook.
# %pip install -r requirements.txt


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
from scipy import stats

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import torch
from torch import nn
from torch.nn.utils import prune
from torch.utils.data import DataLoader, TensorDataset

try:
    import shap
except Exception as exc:
    shap = None
    warnings.warn(f"SHAP could not be imported: {exc}")

try:
    from codecarbon import EmissionsTracker
except Exception:
    EmissionsTracker = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Figure typography only: match the report while preserving all plotting logic/colors.
# The report uses Charis SIL for text and STIX for mathematics.
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Charis SIL", "STIX Two Text", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.titlecolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "legend.labelcolor": "black",
})
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__, "numpy:", np.__version__)
print("scikit-learn imported successfully")
print("LightGBM:", lgb.__version__)
print("PyTorch:", torch.__version__)


In [ ]:
@dataclass
class Config:
    # Data and output locations
    data_dir: Path = Path(os.getenv("ASHRAE_DATA_DIR", next((candidate for candidate in (Path.cwd() / "/kaggle/input/competitions/ashrae-energy-prediction", Path.cwd() / "data" / "ashrae-energy-prediction") if candidate.exists()), Path.cwd() / "ashrae-energy-prediction")))
    output_dir: Path = Path(os.getenv("TROPILITE_OUTPUT_DIR", Path.cwd() / "outputs" / "tropilite_ie"))

    # Reproduction settings
    random_seed: int = 42
    full_data: bool = os.getenv("TROPILITE_FULL_DATA", "0") == "1"
    # Protocol A: the primary target is electricity only (meter == 0).
    electricity_only: bool = os.getenv("TROPILITE_ELECTRICITY_ONLY", "1") == "1"
    # Sampling unit is the BUILDING (complete hourly series retained), not scattered rows,
    # so causal lags and GRU sequence windows are well defined.
    sample_buildings: int = int(os.getenv("TROPILITE_SAMPLE_BUILDINGS", "300"))
    split_mode: str = os.getenv("TROPILITE_SPLIT_MODE", "chronological")

    # Climate labels: either a manual override or inferred from site weather climatology
    # via simplified Koppen rules (Af/Am/Aw). Empty tuple => inference is used.
    tropical_site_ids: Tuple[int, ...] = tuple(
        int(x) for x in os.getenv("TROPILITE_TROPICAL_SITES", "").split(",") if x.strip() != ""
    )

    # Execution controls
    fast_mode: bool = os.getenv("TROPILITE_FAST_MODE", "0") == "1"
    synthetic_if_missing: bool = os.getenv("TROPILITE_SYNTHETIC", "0") == "1"
    run_shap: bool = os.getenv("TROPILITE_RUN_SHAP", "1") == "1"
    run_codecarbon: bool = os.getenv("TROPILITE_CODECARBON", "0") == "1"
    n_jobs: int = max(1, min(8, (os.cpu_count() or 2) - 1))

    # --- Memory-safety controls (added for Kaggle-sized RAM budgets, e.g. 16 GB) ---
    # When on, sample_buildings / GRU eval-window counts / boosting search grids are
    # automatically capped to values that comfortably fit a 16 GB Kaggle session.
    # Set TROPILITE_LOW_MEMORY=0 to run the notebook at its original, uncapped scale
    # (only recommended on a machine with >= 32-64 GB RAM, or a small sample_buildings).
    low_memory_mode: bool = os.getenv("TROPILITE_LOW_MEMORY", "1") == "1"
    # Hard cap on GRU validation/test windows materialized PER BUILDING. The original
    # notebook used an effectively unlimited cap (10**9) here, which for L=168 could
    # materialize several GB of (L, features) windows per building across the whole
    # validation+test set, for all three sequence lengths at once -> OOM. Capping this
    # (and, separately, only ever holding ONE sequence length's windows in memory at a
    # time -- see the GRU cell) is the single biggest fix for the reported crash.
    max_eval_windows_per_building: int = int(os.getenv("TROPILITE_MAX_EVAL_WINDOWS", "600"))
    rf_n_estimators: int = int(os.getenv("TROPILITE_RF_ESTIMATORS", "300"))

    # Neural settings (genuine sequence branch)
    seq_lengths: Tuple[int, ...] = (24, 48, 168)
    gru_hidden: int = 32
    n_seeds: int = 5                     # >= 5 seeds required by the manuscript
    max_train_windows_per_building: int = 96
    epochs: int = 30
    batch_size: int = 512
    patience: int = 5
    learning_rate_nn: float = 1e-3

    # Tree settings
    lgb_learning_rate: float = 0.05
    lgb_num_leaves_grid: Tuple[int, ...] = (31, 63)
    lgb_feature_fraction_grid: Tuple[float, ...] = (0.8, 1.0)
    lgb_min_child_samples_grid: Tuple[int, ...] = (20, 50)
    lgb_bagging_fraction_grid: Tuple[float, ...] = (1.0, 0.8)
    boost_rounds: int = 1000
    early_stopping_rounds: int = 50
    xgb_cat_learning_rate: float = 0.05
    xgb_cat_max_depth: int = 6

    # Fusion gate: retain the GRU fusion only when it beats the strongest tree model
    # on validation by at least this relative RMSE improvement, consistently across seeds.
    fusion_min_improvement: float = 0.01
    fusion_seed_consistency: float = 0.8   # fraction of seeds that must agree in sign
    pruning_amount: float = 0.40

    # EDA/output controls
    shap_sample_size: int = 500
    bootstrap_iterations: int = 1000
    block_length_hours: int = 168

    def finalize(self):
        if self.fast_mode:
            self.sample_buildings = min(self.sample_buildings, 8)
            self.epochs = min(self.epochs, 3)
            self.batch_size = min(self.batch_size, 256)
            self.boost_rounds = min(self.boost_rounds, 60)
            self.early_stopping_rounds = min(self.early_stopping_rounds, 10)
            self.bootstrap_iterations = min(self.bootstrap_iterations, 100)
            self.shap_sample_size = min(self.shap_sample_size, 150)
            self.max_train_windows_per_building = min(self.max_train_windows_per_building, 16)
            self.n_seeds = min(self.n_seeds, 2)

        if self.low_memory_mode and not self.fast_mode:
            # Conservative caps chosen to keep peak RSS comfortably under ~13 GB on a
            # standard 16 GB Kaggle CPU session. Override any of these via the matching
            # TROPILITE_* environment variable if more RAM is available.
            self.sample_buildings = min(self.sample_buildings, 150)
            self.max_eval_windows_per_building = min(self.max_eval_windows_per_building, 400)
            self.rf_n_estimators = min(self.rf_n_estimators, 150)
            self.lgb_min_child_samples_grid = (self.lgb_min_child_samples_grid[0],)
            self.lgb_bagging_fraction_grid = (self.lgb_bagging_fraction_grid[0],)
            self.shap_sample_size = min(self.shap_sample_size, 300)
            self.bootstrap_iterations = min(self.bootstrap_iterations, 500)

        self.output_dir.mkdir(parents=True, exist_ok=True)
        (self.output_dir / "figures").mkdir(exist_ok=True)
        (self.output_dir / "models").mkdir(exist_ok=True)
        (self.output_dir / "tables").mkdir(exist_ok=True)
        return self

cfg = Config().finalize()
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, indent=2))
if cfg.low_memory_mode:
    print("\nlow_memory_mode is ON: sample_buildings, GRU eval-window counts, and search\n"
          "grids have been capped for a ~16 GB Kaggle session. Set the environment variable\n"
          "TROPILITE_LOW_MEMORY=0 before creating Config() to run at the original scale.")

if cfg.split_mode not in {"paper_random", "group_building", "chronological"}:
    raise ValueError("split_mode must be paper_random, group_building, or chronological")

# Preregistration snapshot: config hash written BEFORE any model runs (design doc, section 6).
import hashlib
_cfg_dump = json.dumps(
    {k: (list(v) if isinstance(v, tuple) else str(v) if isinstance(v, Path) else v)
     for k, v in asdict(cfg).items()},
    sort_keys=True, default=str)
_prereg = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "config_sha256": hashlib.sha256(_cfg_dump.encode("utf-8")).hexdigest(),
    "note": "Written before any training run. Commit this file to the repo (dated commit) "
            "to support the pre-registered protocol claim.",
    "config": json.loads(_cfg_dump),
}
with open(cfg.output_dir / "preregistration_config.json", "w", encoding="utf-8") as _f:
    json.dump(_prereg, _f, indent=2)
print("Preregistration snapshot written; config sha256 =", _prereg["config_sha256"][:16], "...")


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_global_seed(cfg.random_seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", DEVICE)


def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Memory reduction adapted from the supplied EDA notebook."""
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            for dtype in (np.int8, np.int16, np.int32, np.int64):
                info = np.iinfo(dtype)
                if c_min >= info.min and c_max <= info.max:
                    df[col] = df[col].astype(dtype)
                    break
        elif pd.api.types.is_float_dtype(col_type):
            # float32 is safer than float16 for psychrometric calculations.
            df[col] = pd.to_numeric(df[col], downcast="float")
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB ({100*(start_mem-end_mem)/max(start_mem,1e-9):.1f}% reduction)")
    return df


@contextmanager
def optional_emissions_tracker(run_name: str):
    tracker = None
    if cfg.run_codecarbon and EmissionsTracker is not None:
        tracker = EmissionsTracker(project_name=run_name, output_dir=str(cfg.output_dir), log_level="error")
        tracker.start()
    try:
        yield
    finally:
        if tracker is not None:
            emissions = tracker.stop()
            print(f"Estimated emissions for {run_name}: {emissions:.6f} kg CO2eq")


# --- Memory-hygiene helpers -------------------------------------------------
# Kaggle CPU sessions are typically capped around 16 GB RSS. The three helpers
# below are used throughout the rest of the notebook to (1) report current
# process memory so it is easy to see where usage grows, (2) explicitly drop
# large intermediate objects and force garbage collection, and (3) stop
# matplotlib from accumulating open Figure objects (every unclosed fig from
# plt.show() stays resident until the kernel restarts).

def mem_report(label: str = "") -> float:
    """Print current process RSS in MB and return it (handy as a progress trace)."""
    rss_mb = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
    avail_mb = psutil.virtual_memory().available / (1024**2)
    print(f"[mem] {label:<40s} RSS={rss_mb:8.0f} MB   system available={avail_mb:8.0f} MB")
    return rss_mb


def free(*names: str) -> None:
    """Delete the given variable names from the notebook's global namespace (if
    present) and force a garbage-collection + CUDA cache clear. Safe to call
    with names that don't exist."""
    g = globals()
    for name in names:
        if name in g:
            del g[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def show_and_close(fig=None) -> None:
    """Render the current (or given) figure and immediately close it so it is
    released from matplotlib's internal figure registry. Without this, every
    plt.show() in a long notebook leaves its Figure (and all of its artists /
    backing arrays) resident in memory for the rest of the kernel's life."""
    plt.show()
    if fig is not None:
        plt.close(fig)
    else:
        plt.close("all")

# --- Vector artwork settings (Elsevier/ENB submission) ----------------------
# Save every manuscript figure as PDF (vector) in addition to PNG. Type-42
# (TrueType) fonts keep text as real text and are the Elsevier-preferred
# embedding for PDF/EPS artwork.
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


## 1. Data acquisition and loading

Place the official ASHRAE files in the configured directory. For a code-only smoke test, set `TROPILITE_SYNTHETIC=1`; the generated data are **not** suitable for paper results.


In [ ]:
def create_synthetic_ashrae_dataset(data_dir: Path, seed: int = 42) -> None:
    """Create ASHRAE-shaped data solely to test notebook execution (never for results)."""
    rng = np.random.default_rng(seed)
    data_dir.mkdir(parents=True, exist_ok=True)

    n_sites = 16
    n_buildings = 96
    n_hours = 2160  # 90 days, long enough for 168-hour sequence windows
    timestamps = pd.date_range("2016-01-01", periods=n_hours, freq="h")
    primary_uses = ["Education", "Office", "Entertainment/public assembly", "Lodging/residential", "Public services", "Other"]

    site_ids = np.tile(np.arange(n_sites), math.ceil(n_buildings / n_sites))[:n_buildings]
    rng.shuffle(site_ids)
    meta = pd.DataFrame({
        "site_id": site_ids,
        "building_id": np.arange(n_buildings),
        "primary_use": rng.choice(primary_uses, n_buildings),
        "square_feet": rng.integers(5_000, 500_000, n_buildings),
        "year_built": rng.choice([np.nan, *range(1950, 2016)], n_buildings),
        "floor_count": rng.choice([np.nan, *range(1, 30)], n_buildings),
    })

    weather_parts = []
    for site in range(n_sites):
        tropical = site in (13, 15)
        hour = np.arange(n_hours)
        base_temp = 29.0 if tropical else 15.0 + 4.0 * np.sin(2 * np.pi * hour / (24 * 30))
        air = base_temp + 4.0 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 1.2, n_hours)
        dew = air - (2.0 if tropical else 7.0) + rng.normal(0, 0.8, n_hours)
        precip = (rng.gamma(0.4, 8.0, n_hours) if tropical else rng.gamma(0.3, 2.0, n_hours)).clip(0)
        weather_parts.append(pd.DataFrame({
            "site_id": site, "timestamp": timestamps,
            "air_temperature": air.round(1), "dew_temperature": dew.round(1),
            "precip_depth_1_hr": precip.round(1),
            "sea_level_pressure": rng.normal(1013, 5, n_hours).round(1),
            "wind_speed": rng.gamma(2, 2, n_hours).round(1),
            "cloud_coverage": rng.choice([0, 1, 2, 3, 4], n_hours).astype(float),
            "wind_direction": rng.integers(0, 360, n_hours).astype(float),
        }))
    weather = pd.concat(weather_parts, ignore_index=True)

    meter_rows = []
    for b in range(n_buildings):
        site = int(meta.loc[b, "site_id"])
        temp = weather.loc[weather["site_id"] == site, "air_temperature"].to_numpy()
        hour = np.arange(n_hours)
        base = rng.uniform(50, 400)
        load = base * (1 + 0.4 * np.sin(2 * np.pi * (hour - 8) / 24)) * (1 + 0.02 * (temp - temp.mean()))
        load = np.clip(load + rng.normal(0, base * 0.05, n_hours), 0, None)
        meter_rows.append(pd.DataFrame({
            "building_id": b, "meter": 0, "timestamp": timestamps,
            "meter_reading": np.round(np.expm1(np.log1p(load)), 4),
        }))
        if b % 5 == 0:  # a minority of buildings also have a chilled-water meter
            meter_rows.append(pd.DataFrame({
                "building_id": b, "meter": 1,
                "timestamp": rng.choice(timestamps, size=72, replace=False),
                "meter_reading": rng.gamma(2, 100, 72).round(4),
            }))
    train = pd.concat(meter_rows, ignore_index=True)

    meta.to_csv(data_dir / "building_metadata.csv", index=False)
    weather.to_csv(data_dir / "weather_train.csv", index=False)
    train.to_csv(data_dir / "train.csv", index=False)
    print(f"Synthetic dataset written to {data_dir}: {len(train)} meter rows, {n_buildings} buildings.")


In [ ]:
def create_synthetic_ashrae_dataset(data_dir: Path, seed: int = 42) -> None:
    """Create ASHRAE-shaped data solely to test notebook execution (never for results)."""
    rng = np.random.default_rng(seed)
    data_dir.mkdir(parents=True, exist_ok=True)

    n_sites = 16
    n_buildings = 96
    n_hours = 2160  # 90 days, long enough for 168-hour sequence windows
    timestamps = pd.date_range("2016-01-01", periods=n_hours, freq="h")
    primary_uses = ["Education", "Office", "Entertainment/public assembly", "Lodging/residential", "Public services", "Other"]

    site_ids = np.tile(np.arange(n_sites), math.ceil(n_buildings / n_sites))[:n_buildings]
    rng.shuffle(site_ids)
    meta = pd.DataFrame({
        "site_id": site_ids,
        "building_id": np.arange(n_buildings),
        "primary_use": rng.choice(primary_uses, n_buildings),
        "square_feet": rng.integers(5_000, 500_000, n_buildings),
        "year_built": rng.choice([np.nan, *range(1950, 2016)], n_buildings),
        "floor_count": rng.choice([np.nan, *range(1, 30)], n_buildings),
    })

    weather_parts = []
    for site in range(n_sites):
        tropical = site in (13, 15)
        hour = np.arange(n_hours)
        base_temp = 29.0 if tropical else 15.0 + 4.0 * np.sin(2 * np.pi * hour / (24 * 30))
        air = base_temp + 4.0 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 1.2, n_hours)
        dew = air - (2.0 if tropical else 7.0) + rng.normal(0, 0.8, n_hours)
        precip = (rng.gamma(0.4, 8.0, n_hours) if tropical else rng.gamma(0.3, 2.0, n_hours)).clip(0)
        weather_parts.append(pd.DataFrame({
            "site_id": site, "timestamp": timestamps,
            "air_temperature": air.round(1), "dew_temperature": dew.round(1),
            "precip_depth_1_hr": precip.round(1),
            "sea_level_pressure": rng.normal(1013, 5, n_hours).round(1),
            "wind_speed": rng.gamma(2, 2, n_hours).round(1),
            "cloud_coverage": rng.choice([0, 1, 2, 3, 4], n_hours).astype(float),
            "wind_direction": rng.integers(0, 360, n_hours).astype(float),
        }))
    weather = pd.concat(weather_parts, ignore_index=True)

    meter_rows = []
    for b in range(n_buildings):
        site = int(meta.loc[b, "site_id"])
        temp = weather.loc[weather["site_id"] == site, "air_temperature"].to_numpy()
        hour = np.arange(n_hours)
        base = rng.uniform(50, 400)
        load = base * (1 + 0.4 * np.sin(2 * np.pi * (hour - 8) / 24)) * (1 + 0.02 * (temp - temp.mean()))
        load = np.clip(load + rng.normal(0, base * 0.05, n_hours), 0, None)
        meter_rows.append(pd.DataFrame({
            "building_id": b, "meter": 0, "timestamp": timestamps,
            "meter_reading": np.round(np.expm1(np.log1p(load)), 4),
        }))
        if b % 5 == 0:  # a minority of buildings also have a chilled-water meter
            meter_rows.append(pd.DataFrame({
                "building_id": b, "meter": 1,
                "timestamp": rng.choice(timestamps, size=72, replace=False),
                "meter_reading": rng.gamma(2, 100, 72).round(4),
            }))
    train = pd.concat(meter_rows, ignore_index=True)

    meta.to_csv(data_dir / "building_metadata.csv", index=False)
    weather.to_csv(data_dir / "weather_train.csv", index=False)
    train.to_csv(data_dir / "train.csv", index=False)
    print(f"Synthetic dataset written to {data_dir}: {len(train)} meter rows, {n_buildings} buildings.")


def exact_proportional_sample_buildings(building_table: pd.DataFrame, strata_col: str, n: int, seed: int) -> pd.DataFrame:
    """Sample exactly n BUILDINGS while preserving stratum proportions as closely as possible."""
    if n <= 0 or n >= len(building_table):
        return building_table.copy()

    counts = building_table[strata_col].value_counts(dropna=False)
    raw = counts / counts.sum() * n
    quota = np.floor(raw).astype(int)
    quota = np.minimum(quota, counts)
    remainder = n - int(quota.sum())
    if remainder > 0:
        fractional = (raw - quota).sort_values(ascending=False)
        for key in fractional.index:
            if remainder == 0:
                break
            if quota.loc[key] < counts.loc[key]:
                quota.loc[key] += 1
                remainder -= 1

    picks = []
    for key, q in quota.items():
        if q <= 0:
            continue
        group = building_table[building_table[strata_col] == key]
        picks.append(group.sample(n=int(q), random_state=seed))
    return pd.concat(picks, ignore_index=True)


def infer_site_climate(weather: pd.DataFrame) -> pd.DataFrame:
    """Assign simplified Koppen tropical classes (Af/Am/Aw vs other) from each site's
    own weather climatology. Reproducible from the supplied weather files; final
    submission should still publish coordinates + a raster lookup for verification."""
    w = weather.copy()
    w["timestamp"] = pd.to_datetime(w["timestamp"], errors="coerce")
    w["month"] = w["timestamp"].dt.month
    rows = []
    for site, g in w.groupby("site_id"):
        temp = pd.to_numeric(g["air_temperature"], errors="coerce")
        precip = pd.to_numeric(g["precip_depth_1_hr"], errors="coerce")
        monthly_t = temp.groupby(g["month"]).mean()
        monthly_p = precip.groupby(g["month"]).sum()
        t_all = monthly_t.dropna()
        p_all = monthly_p.dropna()
        n_months_common = len(set(t_all.index) & set(p_all.index))
        koppen = "Undefined"
        if len(t_all) >= 12 and n_months_common >= 12:
            p_all = p_all.reindex(sorted(p_all.index))
            t_all = t_all.reindex(sorted(t_all.index))
            if (t_all >= 18).all():
                annual_p = p_all.sum()
                driest = p_all.min()
                if driest >= 60:
                    koppen = "Af"
                elif driest >= 100 - annual_p / 25:
                    koppen = "Am"
                else:
                    koppen = "Aw"
            else:
                koppen = "Non-tropical"
        rows.append({"site_id": int(site), "koppen_class": koppen,
                     "mean_air_temp_c": float(t_all.mean()) if len(t_all) else np.nan,
                     "annual_precip_mm_est": float(p_all.sum()) if len(p_all) else np.nan})
    return pd.DataFrame(rows).sort_values("site_id").reset_index(drop=True)


def load_and_sample_data(cfg):
    if cfg.synthetic_if_missing and not (cfg.data_dir / "train.csv").exists():
        print("Real data not found and TROPILITE_SYNTHETIC=1; generating ASHRAE-shaped "
              "synthetic data FOR EXECUTION TESTING ONLY (never for reported results).")
        create_synthetic_ashrae_dataset(cfg.data_dir, seed=cfg.random_seed)
    print("Loading building metadata and weather...")
    meta = pd.read_csv(cfg.data_dir / "building_metadata.csv")
    weather = pd.read_csv(cfg.data_dir / "weather_train.csv")

    print("Loading meter readings...")
    train = pd.read_csv(
        cfg.data_dir / "train.csv",
        usecols=["building_id", "meter", "timestamp", "meter_reading"],
    )
    n_raw = len(train)
    if cfg.electricity_only:
        train = train.loc[train["meter"] == 0].copy()
        print(f"Electricity-only filter: {n_raw} -> {len(train)} rows (meter == 0 retained).")
    if train.empty:
        raise ValueError("No electricity rows found; check the electricity_only setting.")

    # Attach the fields required for building-level stratification.
    strat_meta = meta[["building_id", "site_id", "primary_use"]].copy()
    train = train.merge(strat_meta, on="building_id", how="left", validate="many_to_one")

    # Climate labels: manual override wins, otherwise infer Koppen classes from weather.
    if cfg.tropical_site_ids:
        print("Using manually configured tropical sites:", cfg.tropical_site_ids)
        climate_table = pd.DataFrame({
            "site_id": sorted(meta["site_id"].unique()),
            "koppen_class": ["Manual"] * meta["site_id"].nunique(),
        })
        climate_table["koppen_class"] = climate_table["site_id"].isin(cfg.tropical_site_ids).map({True: "ManualTropical", False: "ManualOther"})
    else:
        climate_table = infer_site_climate(weather)
        inferred = tuple(int(s) for s in climate_table.loc[climate_table["koppen_class"].isin(["Af", "Am", "Aw"]), "site_id"])
        object.__setattr__(cfg, "tropical_site_ids", inferred)
        print("Inferred tropical sites (Af/Am/Aw):", inferred)
    coords_path = cfg.data_dir / "site_coordinates.csv"
    if coords_path.exists():
        coords = pd.read_csv(coords_path)
        climate_table = climate_table.merge(coords, on="site_id", how="left", validate="one_to_one")
        climate_table["classification_basis"] = np.where(
            climate_table["koppen_class"].isin(["ManualTropical", "ManualOther"]),
            "manual override", "weather climatology (>=12 months)")
        print("Site coordinates attached; for the final submission cite the pinned "
              "Beck et al. raster lookup against these coordinates.")
    else:
        print("Optional site_coordinates.csv (site_id,latitude,longitude) not found; "
              "add it so the climate classification is reviewer-verifiable.")
    climate_table.to_csv(cfg.output_dir / "site_climate_map.csv", index=False)

    train["is_tropical"] = train["site_id"].isin(cfg.tropical_site_ids).astype("int8")
    climate_lookup = climate_table.set_index("site_id")["koppen_class"]
    train["climate_stratum"] = train["site_id"].map(climate_lookup).fillna("Unknown")

    # Building-level table for stratified building sampling (whole series per building).
    building_table = (
        train.groupby("building_id")
        .agg(site_id=("site_id", "first"), primary_use=("primary_use", "first"),
             climate_stratum=("climate_stratum", "first"),
             n_rows=("meter_reading", "size"), is_tropical=("is_tropical", "max"))
        .reset_index()
    )
    building_table["sample_stratum"] = (
        building_table["primary_use"].fillna("Unknown").astype(str)
        + "|" + building_table["climate_stratum"].astype(str)
    )
    n_buildings_target = min(cfg.sample_buildings, len(building_table))
    sampled_buildings = exact_proportional_sample_buildings(
        building_table, "sample_stratum", n_buildings_target, cfg.random_seed
    )
    df = train[train["building_id"].isin(sampled_buildings["building_id"])].copy()
    print(f"Sampled {len(sampled_buildings)} buildings -> {len(df)} hourly rows "
          f"(complete series retained per building).")

    # Merge complete metadata and weather (drop stratification-only columns first to
    # avoid site_id/primary_use duplication on the metadata merge).
    df = df.drop(columns=["site_id", "primary_use"], errors="ignore")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")
    df = df.merge(meta, on="building_id", how="left", validate="many_to_one")
    df = df.merge(weather, on=["site_id", "timestamp"], how="left", validate="many_to_one")

    meta = reduce_mem_usage(meta)
    weather = reduce_mem_usage(weather)
    df = reduce_mem_usage(df)
    gc.collect()
    return df, meta, weather, climate_table


df_raw, meta, weather, SITE_CLIMATE_TABLE = load_and_sample_data(cfg)
print("Merged shape:", df_raw.shape)
print("Sites:", df_raw["site_id"].nunique(), "Buildings:", df_raw["building_id"].nunique())
print("Meters present:", sorted(df_raw["meter"].unique().tolist()))
print("Tropical share:", f"{100*df_raw['site_id'].isin(cfg.tropical_site_ids).mean():.2f}%")
df_raw.head()
mem_report("after loading + sampling raw data")


## 2. Exploratory data analysis (following and extending the supplied EDA notebook)


In [ ]:
# Dataset coverage
coverage = pd.DataFrame({
    "rows": [len(df_raw)],
    "buildings": [df_raw["building_id"].nunique()],
    "sites": [df_raw["site_id"].nunique()],
    "primary_uses": [df_raw["primary_use"].nunique(dropna=True)],
    "tropical_rows": [int(df_raw["site_id"].isin(cfg.tropical_site_ids).sum())],
})
display(coverage)

use_dist = df_raw["primary_use"].value_counts(normalize=True).mul(100).rename("Percent").to_frame()
site_dist = df_raw["site_id"].value_counts().sort_index().rename("Rows").to_frame()
display(use_dist.head(15), site_dist)


In [ ]:
# Missing-value overview
metadata_missing = meta.isna().sum().sort_values(ascending=False)
weather_missing = weather.isna().sum().sort_values(ascending=False)

display(metadata_missing.to_frame("Missing"), weather_missing.to_frame("Missing"))

fig, ax = plt.subplots(figsize=(10, 5))
metadata_missing.plot(kind="bar", ax=ax)
ax.set_title("Missing Values in Building Metadata")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "missing_metadata.png", dpi=180, bbox_inches="tight")
show_and_close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
weather_missing.plot(kind="bar", ax=ax)
ax.set_title("Missing Values in Weather Data")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "missing_weather.png", dpi=180, bbox_inches="tight")
show_and_close(fig)


In [ ]:
# Target distribution
log_target_eda = np.log1p(df_raw["meter_reading"].clip(lower=0)).dropna().to_numpy()
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(log_target_eda, bins=50, density=False, alpha=0.8)
if len(np.unique(log_target_eda)) > 1:
    kde = stats.gaussian_kde(log_target_eda)
    grid = np.linspace(log_target_eda.min(), log_target_eda.max(), 300)
    # Scale density to approximate histogram counts.
    bin_width = (log_target_eda.max() - log_target_eda.min()) / 50
    ax.plot(grid, kde(grid) * len(log_target_eda) * bin_width)
ax.set_title("Log-transformed Meter Reading Distribution")
ax.set_xlabel("log(1 + meter_reading)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "meter_reading_log.png", dpi=180, bbox_inches="tight")
show_and_close(fig)


In [ ]:
# Air temperature versus target
plot_sample = df_raw.sample(n=min(10000, len(df_raw)), random_state=cfg.random_seed)
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=plot_sample,
    x="air_temperature",
    y=np.log1p(plot_sample["meter_reading"].clip(lower=0)),
    hue=plot_sample["site_id"].isin(cfg.tropical_site_ids).map({True: "Tropical", False: "Other"}),
    alpha=0.25,
    s=18,
    ax=ax,
)
ax.set_title("Air Temperature vs Log Meter Reading")
ax.set_ylabel("log(1 + meter_reading)")
ax.legend(title="Climate subset")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "temp_vs_meter.png", dpi=180, bbox_inches="tight")
show_and_close(fig)


In [ ]:
# Correlation matrix, following the original EDA and adding humidity/pressure variables when present.
corr_cols = [
    c for c in [
        "meter_reading", "square_feet", "air_temperature", "dew_temperature",
        "sea_level_pressure", "wind_speed", "cloud_coverage", "precip_depth_1_hr"
    ] if c in df_raw.columns
]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_raw[corr_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "correlation_matrix.png", dpi=180, bbox_inches="tight")
show_and_close(fig)


## 3A. Verified climate classification (C2/M1 repair)

Create a reproducible site climate table.

Required columns:
- site_id
- latitude
- longitude
- Köppen–Geiger class
- tropical flag
- source

Tropical classes:
Af, Am, Aw.

H1 is evaluated only when at least two verified tropical sites exist.

In [ ]:
# Render and export the verified site climate table (C2/M1 repair).
def build_site_climate_table(climate_map: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    table = climate_map.copy()
    table["tropical_flag"] = table["koppen_class"].isin(["Af", "Am", "Aw", "ManualTropical"])
    table.to_csv(output_path, index=False)
    return table

climate_table = build_site_climate_table(SITE_CLIMATE_TABLE, cfg.output_dir / "tables" / "site_climate_map.csv")
display(climate_table)
print("Tropical sites used throughout (Af/Am/Aw or manual override):", cfg.tropical_site_ids)
if len(cfg.tropical_site_ids) < 2:
    warnings.warn(
        "Fewer than two tropical sites were identified. Per the protocol, H1/H2 tropical "
        "analyses must be reported as 'not testable' and no climate-effect claims may be made."
    )


## Protocol A model grid

Primary ablations:
- LightGBM Base
- LightGBM + Enthalpy
- LightGBM + Lag
- LightGBM + Enthalpy + Lag

Secondary models:
- Ridge
- Random Forest
- XGBoost
- CatBoost

GRU:
- sequence length L = 24, 48, 168;
- weather + load history + calendar + enthalpy;
- 5 random seeds.

## 3. Physics-informed feature engineering and lag construction


In [ ]:
WEATHER_COLUMNS = [
    "air_temperature", "dew_temperature", "sea_level_pressure", "wind_speed",
    "cloud_coverage", "precip_depth_1_hr", "wind_direction"
]


def add_physics_and_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out["target_log"] = np.log1p(out["meter_reading"].clip(lower=0)).astype("float32")

    out["hour"] = out["timestamp"].dt.hour.astype("int8")
    out["dayofweek"] = out["timestamp"].dt.dayofweek.astype("int8")
    out["month"] = out["timestamp"].dt.month.astype("int8")
    out["quarter"] = out["timestamp"].dt.quarter.astype("int8")

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["dayofweek"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dayofweek"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * (out["month"] - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (out["month"] - 1) / 12)

    # Stable integer encoding; category dictionary is exported with the model bundle.
    primary_use_categories = sorted(out["primary_use"].fillna("Unknown").astype(str).unique())
    mapping = {name: idx for idx, name in enumerate(primary_use_categories)}
    out["primary_use_code"] = out["primary_use"].fillna("Unknown").astype(str).map(mapping).astype("int16")
    out.attrs["primary_use_mapping"] = mapping

    # Paper equations: input temperatures clipped to [-50, 60] °C.
    T = pd.to_numeric(out["air_temperature"], errors="coerce").clip(-50, 60)
    Tdew = pd.to_numeric(out["dew_temperature"], errors="coerce").clip(-50, 60)
    vapor_pressure_hpa = 6.11 * np.power(10.0, (7.5 * Tdew) / (237.3 + Tdew))
    denominator = np.maximum(1013.25 - vapor_pressure_hpa, 1.0)
    humidity_ratio = 0.622 * vapor_pressure_hpa / denominator
    enthalpy = 1.006 * T + humidity_ratio * (2501 + 1.86 * T)

    out["vapor_pressure_hpa"] = vapor_pressure_hpa.astype("float32")
    out["humidity_ratio"] = humidity_ratio.astype("float32")
    out["physics_enthalpy"] = enthalpy.astype("float32")
    out["is_tropical"] = out["site_id"].isin(cfg.tropical_site_ids).astype("int8")
    return out


def add_target_lags(df: pd.DataFrame, lags: Sequence[int] = (1, 2, 24, 168)) -> pd.DataFrame:
    """Past-only target-history features within each building+meter stream.

    Leakage note: lag values are produced by positional shift on the chronologically
    ordered per-building stream. With the Protocol A chronological split, every lag
    feeding a validation/test row is that row's own past (earlier timestamps), which
    is available at forecast time for one-step-ahead prediction. Cell 22 asserts the
    per-building train -> validation -> test ordering that this property relies on.
    """
    out = df.sort_values(["building_id", "meter", "timestamp"]).copy()
    grouped = out.groupby(["building_id", "meter"], sort=False)["target_log"]
    for lag in lags:
        out[f"meter_log_lag_{lag}"] = grouped.shift(lag).astype("float32")
    # Past-only rolling statistics: shift(1) guarantees the window excludes the current row.
    for window in (24, 168):
        out[f"meter_roll_mean_{window}"] = grouped.transform(
            lambda s: s.shift(1).rolling(window, min_periods=max(4, window // 4)).mean()
        ).astype("float32")
        out[f"meter_roll_std_{window}"] = grouped.transform(
            lambda s: s.shift(1).rolling(window, min_periods=max(4, window // 4)).std()
        ).astype("float32")
    return out.sort_index()


df = add_physics_and_time_features(df_raw)
primary_use_mapping = df.attrs.get("primary_use_mapping", {})
df = add_target_lags(df, (1, 2, 24, 168))
free("df_raw")  # Free up memory as df_raw is no longer needed

physics_summary = df[["air_temperature", "dew_temperature", "humidity_ratio", "physics_enthalpy"]].describe().T
display(physics_summary)
print("Primary-use mapping:", primary_use_mapping)

# Rows without sufficient history (head-of-stream rows whose 168h lag is undefined)
# are dropped, per the protocol. With the per-building chronological split these are
# exclusively early TRAIN rows; the count is documented for the manuscript.
_n_before = len(df)
df = df[df["meter_log_lag_168"].notna()].copy()
df = df.reset_index(drop=True)  # keep row labels == positions for the split machinery
_n_dropped = _n_before - len(df)
print(f"Dropped {_n_dropped} rows with insufficient history "
      f"({_n_before} -> {len(df)}); these are head-of-stream TRAIN rows only.")
with open(cfg.output_dir / "tables" / "dropped_head_rows.txt", "w") as _f:
    _f.write("rows_before=%d" % _n_before + chr(10))
    _f.write("rows_dropped_insufficient_history=%d" % _n_dropped + chr(10))
mem_report("after feature engineering + lag construction")


In [ ]:
LAG_COLUMNS = [
    "meter_log_lag_1", "meter_log_lag_2", "meter_log_lag_24", "meter_log_lag_168",
    "meter_roll_mean_24", "meter_roll_std_24", "meter_roll_mean_168", "meter_roll_std_168",
]
LAG_COLUMNS = [c for c in LAG_COLUMNS if c in df.columns]

BASE_FEATURES = [
    "building_id", "site_id", "primary_use_code", "square_feet",
    "hour", "dayofweek", "month",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "air_temperature", "dew_temperature", "wind_speed", "cloud_coverage",
    "precip_depth_1_hr", "sea_level_pressure", "wind_direction",
]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]
ENTHALPY_FEATURES = BASE_FEATURES + ["vapor_pressure_hpa", "humidity_ratio", "physics_enthalpy"]
LAG_FEATURES = BASE_FEATURES + LAG_COLUMNS
ENTHALPY_LAG_FEATURES = ENTHALPY_FEATURES + LAG_COLUMNS
# Sequence branch input: calendar + weather + physics (+ implicit load history is NOT
# included; the GRU receives the raw past target channel separately below).
SEQUENCE_FEATURES = [c for c in ENTHALPY_FEATURES if c not in ("building_id", "site_id")]

feature_sets = {
    "base": BASE_FEATURES,
    "enthalpy": ENTHALPY_FEATURES,
    "lag": LAG_FEATURES,
    "enthalpy_lag": ENTHALPY_LAG_FEATURES,
}
for name, cols in feature_sets.items():
    print(f"{name:14s}: {len(cols)} features")
print(f"{'sequence':14s}: {len(SEQUENCE_FEATURES)} features + past-target channel")


## 4. Protocol A chronological split and fold-safe feature construction

The previous random split is replaced.

For every building:
- order observations by timestamp;
- earliest 70% → training;
- next 15% → validation;
- latest 15% → testing.

Lag variables:
- y(t-1), y(t-2), y(t-24), y(t-168)
- 24-hour rolling mean/std
- 168-hour rolling mean/std

All source timestamps must be earlier than the forecast origin.

In [ ]:
def collapse_rare_strata(labels: pd.Series, minimum_count: int = 3) -> pd.Series:
    counts = labels.value_counts()
    return labels.where(labels.map(counts) >= minimum_count, "RARE")


def make_splits(df: pd.DataFrame, cfg: Config) -> Dict[str, np.ndarray]:
    idx = np.arange(len(df))

    if cfg.split_mode == "paper_random":
        labels = (
            df["primary_use"].fillna("Unknown").astype(str)
            + "|T" + df["is_tropical"].astype(str)
            + "|Q" + df["quarter"].astype(str)
        )
        labels = collapse_rare_strata(labels, minimum_count=4)

        trainval_idx, test_idx = train_test_split(
            idx, test_size=0.10, random_state=cfg.random_seed, stratify=labels
        )
        train_idx, val_idx = train_test_split(
            trainval_idx,
            test_size=0.20,
            random_state=cfg.random_seed,
            stratify=labels.iloc[trainval_idx],
        )

    elif cfg.split_mode == "group_building":
        splitter = GroupShuffleSplit(
            n_splits=1, test_size=0.10, random_state=cfg.random_seed
        )
        trainval_pos, test_pos = next(
            splitter.split(idx, groups=df["building_id"])
        )
        trainval_idx, test_idx = idx[trainval_pos], idx[test_pos]

        splitter2 = GroupShuffleSplit(
            n_splits=1, test_size=0.20, random_state=cfg.random_seed
        )
        tr_pos, va_pos = next(
            splitter2.split(
                trainval_idx,
                groups=df.iloc[trainval_idx]["building_id"]
            )
        )
        train_idx, val_idx = trainval_idx[tr_pos], trainval_idx[va_pos]

    elif cfg.split_mode == "chronological":
        # Protocol A: per-building chronological split.
        # earliest 70% = train, next 15% = validation, latest 15% = test.
        train_idx, val_idx, test_idx = [], [], []
        for _, group in df.groupby("building_id"):
            group = group.sort_values("timestamp")
            building_indices = group.index.to_numpy()
            n = len(building_indices)
            n_train = int(0.70 * n)
            n_val = int(0.15 * n)
            train_idx.extend(building_indices[:n_train])
            val_idx.extend(building_indices[n_train:n_train + n_val])
            test_idx.extend(building_indices[n_train + n_val:])
        # `train_idx`/`val_idx`/`test_idx` are built in groupby("building_id") key
        # order (grouped by building, chronological within each building), NOT
        # sorted ascending by row position across buildings. Several downstream
        # cells build y_val = df.iloc[splits["val"]] etc. and then separately
        # recompute np.searchsorted(np.sort(splits["val"]), some_positions) to
        # align a different (position-sorted) view against it -- that alignment
        # is only valid if splits["val"] is ALREADY position-sorted. Sort here so
        # every downstream consumer of df.iloc[splits[...]] and any position-based
        # rank lookup against splits[...] agree on the same ordering convention.
        train_idx = np.sort(np.asarray(train_idx))
        val_idx = np.sort(np.asarray(val_idx))
        test_idx = np.sort(np.asarray(test_idx))
    else:
        raise ValueError("split_mode must be paper_random, group_building, or chronological")

    return {"train": train_idx, "val": val_idx, "test": test_idx}


splits = make_splits(df, cfg)

for name, ix in splits.items():
    subset = df.iloc[ix]
    print(
        f"{name:5s}: n={len(ix):6d} "
        f"({100*len(ix)/len(df):5.1f}%), "
        f"buildings={subset['building_id'].nunique():4d}, "
        f"tropical={100*subset['is_tropical'].mean():5.2f}%"
    )

# Protocol A leakage verification (real assertions, not a print-only loop).
if cfg.split_mode == "chronological":
    violations = 0
    duplicate_timestamps = 0
    for _, g in df.groupby("building_id"):
        g = g.sort_values("timestamp")
        if cfg.electricity_only and g["timestamp"].duplicated().any():
            duplicate_timestamps += g["timestamp"].duplicated().sum()
        n = len(g)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        tr_end = g["timestamp"].iloc[n_train - 1] if n_train > 0 else None
        va_start = g["timestamp"].iloc[n_train] if n_train < n else None
        va_end = g["timestamp"].iloc[n_train + n_val - 1] if n_train + n_val > n_train and n_train + n_val <= n else None
        te_start = g["timestamp"].iloc[n_train + n_val] if n_train + n_val < n else None
        if tr_end is not None and va_start is not None and tr_end >= va_start:
            violations += 1
        if va_end is not None and te_start is not None and va_end >= te_start:
            violations += 1
    assert violations == 0, f"Chronological ordering violated in {violations} buildings"
    assert duplicate_timestamps == 0, f"Duplicate timestamps found: {duplicate_timestamps}"
    print(
        "Chronological per-building split verified for every building: "
        "max(train) < min(validation) <= max(validation) < min(test); no duplicate timestamps."
    )

if cfg.split_mode == "paper_random":
    train_buildings = set(df.iloc[splits["train"]]["building_id"])
    test_buildings = set(df.iloc[splits["test"]]["building_id"])
    print("Building overlap between train and test:", len(train_buildings & test_buildings))


In [ ]:
class MatrixBundle:
    def __init__(self, feature_names, imputer, scaler, X_train, X_val, X_test):
        self.feature_names = feature_names
        self.imputer = imputer
        self.scaler = scaler
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test


def prepare_feature_set(df: pd.DataFrame, columns: Sequence[str], splits: Dict[str, np.ndarray]) -> MatrixBundle:
    columns = list(columns)
    imputer = SimpleImputer(strategy="mean")
    scaler = StandardScaler()
    X_train_raw = df.iloc[splits["train"]][columns].to_numpy(dtype=np.float32)
    X_val_raw = df.iloc[splits["val"]][columns].to_numpy(dtype=np.float32)
    X_test_raw = df.iloc[splits["test"]][columns].to_numpy(dtype=np.float32)

    X_train_imp = imputer.fit_transform(X_train_raw)
    X_val_imp = imputer.transform(X_val_raw)
    X_test_imp = imputer.transform(X_test_raw)
    del X_train_raw, X_val_raw, X_test_raw
    gc.collect()

    X_train = scaler.fit_transform(X_train_imp).astype(np.float32, copy=False)
    del X_train_imp
    X_val = scaler.transform(X_val_imp).astype(np.float32, copy=False)
    del X_val_imp
    X_test = scaler.transform(X_test_imp).astype(np.float32, copy=False)
    del X_test_imp
    gc.collect()
    return MatrixBundle(columns, imputer, scaler, X_train, X_val, X_test)


matrices = {name: prepare_feature_set(df, cols, splits) for name, cols in feature_sets.items()}
y_train = df.iloc[splits["train"]]["target_log"].to_numpy(dtype=np.float32)
y_val = df.iloc[splits["val"]]["target_log"].to_numpy(dtype=np.float32)
y_test = df.iloc[splits["test"]]["target_log"].to_numpy(dtype=np.float32)

test_meta = df.iloc[splits["test"]][["building_id", "site_id", "timestamp", "is_tropical", "meter"]].reset_index(drop=True)
print({name: bundle.X_train.shape for name, bundle in matrices.items()})
mem_report("after building train/val/test design matrices")


## 5. Evaluation utilities and experiment registry


In [ ]:
def regression_metrics(y_true_log: np.ndarray, y_pred_log: np.ndarray) -> Dict[str, float]:
    """Log-scale metrics plus zero-safe and inverse-transform (raw kWh) metrics.

    MAPE is intentionally excluded: zero readings make ordinary percentage errors
    unstable (the manuscript's own justification for WAPE/sMAPE).
    """
    y_true_log = np.asarray(y_true_log, dtype=float)
    y_pred_log = np.asarray(y_pred_log, dtype=float)
    rmse = math.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae = mean_absolute_error(y_true_log, y_pred_log)
    r2 = r2_score(y_true_log, y_pred_log)

    ae_log = np.abs(y_true_log - y_pred_log)
    denom = np.abs(y_true_log)
    nz = denom > 1e-9
    wape = float(ae_log[nz].sum() / denom[nz].sum() * 100) if nz.any() else np.nan
    smape = float(np.mean(2 * ae_log / np.maximum(denom + np.abs(y_pred_log), 1e-9)) * 100)

    # Inverse-transform to the raw kWh scale for practical interpretability.
    y_true_raw = np.expm1(np.clip(y_true_log, 0, None))
    y_pred_raw = np.expm1(np.clip(y_pred_log, 0, None))
    rmse_raw = math.sqrt(mean_squared_error(y_true_raw, y_pred_raw))
    mae_raw = mean_absolute_error(y_true_raw, y_pred_raw)

    return {
        "RMSE_log": rmse, "R2_log": r2, "MAE_log": mae,
        "WAPE_log_percent": wape, "sMAPE_log_percent": smape,
        "RMSE_raw_kwh": rmse_raw, "MAE_raw_kwh": mae_raw,
    }


PRIMARY_METRIC = "RMSE_log"

models: Dict[str, object] = {}
predictions_val: Dict[str, np.ndarray] = {}
predictions_test: Dict[str, np.ndarray] = {}
results: List[Dict[str, float]] = []
train_seconds: Dict[str, float] = {}


def register_result(name: str, y_pred_val: np.ndarray, y_pred_test: np.ndarray, model=None, seconds: float = np.nan):
    predictions_val[name] = np.asarray(y_pred_val).reshape(-1)
    predictions_test[name] = np.asarray(y_pred_test).reshape(-1)
    row = {"Model": name, **regression_metrics(y_test, predictions_test[name]), "Train_seconds": seconds}
    results.append(row)
    train_seconds[name] = seconds
    if model is not None:
        models[name] = model
    print(name, {k: (round(v, 5) if isinstance(v, float) else v) for k, v in row.items()})


def fit_and_register_sklearn(name: str, model, matrix_key: str):
    bundle = matrices[matrix_key]
    start = time.perf_counter()
    with optional_emissions_tracker(name.replace(" ", "_")):
        model.fit(bundle.X_train, y_train)
    seconds = time.perf_counter() - start
    register_result(name, model.predict(bundle.X_val), model.predict(bundle.X_test), model, seconds)
    return model


## 6. Classical and gradient-boosted baselines


In [ ]:
# Ridge Regression and Random Forest (no enthalpy)
ridge = fit_and_register_sklearn("Ridge Regression", Ridge(alpha=1.0), "base")

rf = RandomForestRegressor(
    n_estimators=8 if cfg.fast_mode else cfg.rf_n_estimators,
    random_state=cfg.random_seed,
    n_jobs=cfg.n_jobs,
    max_features="sqrt",
)
rf = fit_and_register_sklearn("Random Forest", rf, "base")


In [ ]:
def _lgb_from_params(params: Dict) -> LGBMRegressor:
    return LGBMRegressor(
        objective="regression",
        metric="rmse",
        learning_rate=cfg.lgb_learning_rate,
        n_estimators=cfg.boost_rounds,
        random_state=cfg.random_seed,
        n_jobs=cfg.n_jobs,
        verbosity=-1,
        **params,
    )


def fit_lightgbm(name: str, matrix_key: str) -> LGBMRegressor:
    """LightGBM with a small equal-budget tuning grid selected on VALIDATION only.

    Every LightGBM variant receives the identical grid and selection rule so that
    the ablation comparison stays credible.
    """
    bundle = matrices[matrix_key]
    best_model, best_val_rmse, best_params = None, np.inf, None
    for num_leaves in cfg.lgb_num_leaves_grid:
        for feat_frac in cfg.lgb_feature_fraction_grid:
            for min_child in cfg.lgb_min_child_samples_grid:
                for bag_frac in cfg.lgb_bagging_fraction_grid:
                    params = {"num_leaves": num_leaves, "feature_fraction": feat_frac,
                              "min_child_samples": min_child, "bagging_fraction": bag_frac,
                              "bagging_freq": 1 if bag_frac < 1.0 else 0}
                    model = _lgb_from_params(params)
                    model.fit(
                        bundle.X_train,
                        y_train,
                        eval_set=[(bundle.X_val, y_val)],
                        eval_metric="rmse",
                        callbacks=[lgb.early_stopping(cfg.early_stopping_rounds, verbose=False)],
                    )
                    val_pred = model.predict(bundle.X_val)
                    val_rmse = math.sqrt(mean_squared_error(y_val, val_pred))
                    if val_rmse < best_val_rmse:
                        best_model, best_val_rmse, best_params = model, val_rmse, params
    print(f"{name}: selected params {best_params} (validation RMSE {best_val_rmse:.5f})")
    start = time.perf_counter()
    test_pred = best_model.predict(bundle.X_test)
    seconds = time.perf_counter() - start + getattr(best_model, "_fit_seconds", 0.0)
    register_result(name, best_model.predict(bundle.X_val), test_pred, best_model, seconds)
    return best_model


def fit_xgboost(name: str, matrix_key: str) -> XGBRegressor:
    bundle = matrices[matrix_key]
    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        learning_rate=cfg.xgb_cat_learning_rate,
        max_depth=cfg.xgb_cat_max_depth,
        n_estimators=cfg.boost_rounds,
        early_stopping_rounds=cfg.early_stopping_rounds,
        tree_method="hist",
        random_state=cfg.random_seed,
        n_jobs=cfg.n_jobs,
    )
    start = time.perf_counter()
    with optional_emissions_tracker(name.replace(" ", "_")):
        model.fit(bundle.X_train, y_train, eval_set=[(bundle.X_val, y_val)], verbose=False)
    seconds = time.perf_counter() - start
    register_result(name, model.predict(bundle.X_val), model.predict(bundle.X_test), model, seconds)
    return model


def fit_catboost(name: str, matrix_key: str) -> CatBoostRegressor:
    bundle = matrices[matrix_key]
    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        learning_rate=cfg.xgb_cat_learning_rate,
        depth=cfg.xgb_cat_max_depth,
        iterations=cfg.boost_rounds,
        random_seed=cfg.random_seed,
        thread_count=cfg.n_jobs,
        verbose=False,
        allow_writing_files=False,
    )
    start = time.perf_counter()
    with optional_emissions_tracker(name.replace(" ", "_")):
        model.fit(
            bundle.X_train,
            y_train,
            eval_set=(bundle.X_val, y_val),
            early_stopping_rounds=cfg.early_stopping_rounds,
            verbose=False,
        )
    seconds = time.perf_counter() - start
    register_result(name, model.predict(bundle.X_val), model.predict(bundle.X_test), model, seconds)
    return model


# No-enthalpy boosting baselines
lgb_base = fit_lightgbm("LightGBM (No Enthalpy)", "base")
xgb_base = fit_xgboost("XGBoost", "base")
cat_base = fit_catboost("CatBoost", "base")

# Enthalpy ablations
lgb_enthalpy = fit_lightgbm("LightGBM + Enthalpy", "enthalpy")
xgb_enthalpy = fit_xgboost("XGBoost + Enthalpy", "enthalpy")
cat_enthalpy = fit_catboost("CatBoost + Enthalpy", "enthalpy")

# Temporal ablation and the combined sensitivity analysis
lgb_lag = fit_lightgbm("LightGBM + Lag Features", "lag")
lgb_enthalpy_lag = fit_lightgbm("LightGBM + Enthalpy + Lag", "enthalpy_lag")

# Tree-family ablation at the lag feature matrix (Section 8.7 of the paper):
# identical features as the deployed LightGBM + Lag model, different booster,
# so the comparison separates the contribution of the lag features from the
# choice of tree family.
xgb_lag = fit_xgboost("XGBoost + Lag Features", "lag")
cat_lag = fit_catboost("CatBoost + Lag Features", "lag")

# Seasonal-naive persistence baselines (Section 8.7 of the paper): the forecast
# IS the observed log target from 1/24/168 hours earlier. No model is fit
# (zero training cost); predictions are evaluated on the identical test rows
# with the identical metric function, so they contextualize absolute accuracy.
for _lag, _pname in ((1, "Persistence (lag-1)"), (24, "Persistence (lag-24)"), (168, "Persistence (lag-168)")):
    _col = f"meter_log_lag_{_lag}"
    register_result(_pname,
                    df.iloc[splits["val"]][_col].to_numpy(dtype=np.float32),
                    df.iloc[splits["test"]][_col].to_numpy(dtype=np.float32),
                    model=None, seconds=0.0)
# (Ridge and Random Forest reference baselines are trained in the preceding cell.)

# --- Memory hygiene: free train/val design matrices now that every classical
# and boosting model has been fit and its predictions cached. Only X_test
# (needed later by SHAP and the latency benchmark) is kept. ---
for _key, _bundle in matrices.items():
    _bundle.X_train = None
    _bundle.X_val = None
del _key, _bundle
gc.collect()
mem_report("after freeing train/val design matrices (post-boosting)")


## 7. Genuine GRU sequence forecasting branch

The GRU uses real temporal windows:
- L ∈ {24, 48, 168};
- past load, weather, calendar, and enthalpy features;
- Huber loss;
- early stopping;
- five seeds with mean ± SD reporting.

Single-timestep GRU experiments are removed.

In [ ]:
# ---------------------------------------------------------------------------
# Genuine multi-hour GRU sequence branch (Protocol A).
# Each training sample is a window of L consecutive past hours ending at t-1,
# predicting y'_t. Train windows lie fully inside the train period; validation
# and test windows may reach back into earlier periods (past data is available
# at forecast time). This replaces the pilot's single-timestep GRU.
#
# MEMORY FIX (this is the main cause of the reported Kaggle OOM / kernel
# restart): the original implementation built dense (n_rows, L, n_features)
# window arrays for EVERY validation/test row (max_windows_per_building was
# effectively unlimited, 10**9) for all three sequence lengths (L=24, 48, 168)
# at once, and kept ALL of them -- plus 5 seeds' worth of duplicate
# model+array references -- alive simultaneously in `gru_runs` for the rest of
# the notebook. For L=168 alone this can easily reach several GB, and with all
# three lengths resident at once (nothing was ever freed) it is easy to blow
# past a 16 GB Kaggle session.
#
# The fix below keeps the exact same modeling logic (same architecture, same
# training loop, same seeds, same selection rule) but changes memory handling:
#   1. `cfg.max_eval_windows_per_building` caps the number of validation/test
#      windows materialized per building (still i.i.d. across buildings, so
#      the reported metrics are a large, representative subsample rather than
#      literally every row -- documented in the printed output below).
#   2. Only ONE sequence length's window arrays are ever in memory at a time.
#      Each seed's trained model is stored as a small CPU state_dict instead
#      of a full model + its own copy of the (large) window arrays.
#   3. After the best sequence length is selected (by mean validation RMSE,
#      exactly as before), its window arrays are rebuilt ONCE and each seed's
#      model is reloaded from its saved state_dict -- this reproduces the
#      original `gru_runs[(L, seed)] = {"model": ..., "X_va": ..., "X_te": ...}`
#      structure that later cells (fusion gate, quantization, benchmarking)
#      already expect, so nothing downstream needs to change.
# ---------------------------------------------------------------------------
SEQ_TARGET_COL = "target_log"
TARGET_CHANNEL_POS = 0  # past log-load is prepended as channel 0


def _seq_feature_matrix(df: pd.DataFrame, feature_cols: Sequence[str]):
    """Chronologically ordered, train-imputed/standardized sequence features."""
    cols = [SEQ_TARGET_COL] + list(feature_cols)
    raw = df[cols].to_numpy(dtype=np.float32)
    tr = np.isin(np.arange(len(df)), splits["train"])
    med = np.nanmedian(raw[tr], axis=0)
    raw = np.where(np.isnan(raw), med, raw)
    mu = raw[tr].mean(axis=0)
    sd = np.maximum(raw[tr].std(axis=0), 1e-6)
    return ((raw - mu) / sd).astype(np.float32)


def build_sequences(df: pd.DataFrame, feature_cols: Sequence[str], L: int, scope: str,
                    max_windows_per_building: int = 96, seed: int = 42):
    """Return (X [n, L, d], y [n], window_end_positions) for the requested scope.

    `max_windows_per_building` is now ALWAYS finite (see cfg.max_eval_windows_per_building
    for validation/test) so a single call can never materialize an unbounded array.
    """
    mat = _seq_feature_matrix(df, feature_cols)
    d = mat.shape[1]
    scope_set = set(np.asarray(splits[scope]).tolist())
    Xs, ys, pos = [], [], []
    rng = np.random.default_rng(seed)
    for _, g_idx in df.groupby("building_id").indices.items():
        g_idx = np.sort(np.asarray(g_idx))
        order = g_idx[np.argsort(df["timestamp"].to_numpy()[g_idx])]
        # `order` is sorted by TIMESTAMP, not by row-position value, so row positions
        # for a building are generally scattered (non-monotonic) across the wider
        # dataframe. np.searchsorted requires a numerically-sorted array and silently
        # returns wrong ranks on a non-monotonic one -- use an explicit position ->
        # rank-within-building-stream map instead.
        rank_of = {int(p): r for r, p in enumerate(order)}
        end_positions = [i for i in order if i in scope_set]
        window_pool = []
        for t_pos, t_global in enumerate(end_positions):
            j = rank_of[int(t_global)]  # position within building stream
            if j >= L:
                window_pool.append((j, t_global))
        if len(window_pool) > max_windows_per_building:
            sel = rng.choice(len(window_pool), size=max_windows_per_building, replace=False)
            window_pool = [window_pool[k] for k in sel]
        for j, t_global in window_pool:
            window_rows = order[j - L:j]
            Xs.append(mat[window_rows])
            ys.append(mat[t_global, 0])  # scaled target; rescaled to log target below
            pos.append(t_global)
    if not Xs:
        return np.empty((0, L, d), dtype=np.float32), np.empty((0,), dtype=np.float32), np.empty((0,), dtype=int)
    X = np.stack(Xs); y = np.asarray(ys, dtype=np.float32); pos = np.asarray(pos, dtype=int)
    o = np.argsort(pos)  # ascending global position: predictions align with split rows
    return X[o], y[o], pos[o]


class GRUSeqRegressor(nn.Module):
    """Compact single-layer GRU -> dense head, per the redesigned protocol."""
    def __init__(self, input_dim: int, hidden_dim: int = 32, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.head(self.dropout(sequence[:, -1, :])).squeeze(-1)


def _seq_loader(X, y=None, shuffle=False):
    X_tensor = torch.from_numpy(X.astype(np.float32))
    dataset = TensorDataset(X_tensor) if y is None else TensorDataset(
        X_tensor, torch.from_numpy(y.astype(np.float32)))
    return DataLoader(dataset, batch_size=cfg.batch_size, shuffle=shuffle, num_workers=0)


def _model_device(model: nn.Module) -> torch.device:
    # Quantized modules expose no plain parameters; next() would raise StopIteration.
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def predict_torch(model: nn.Module, X: np.ndarray) -> np.ndarray:
    model.eval()
    device = _model_device(model)
    preds = []
    with torch.no_grad():
        for (xb,) in _seq_loader(X):
            xb = xb.to(device)
            preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)


def _train_one(model, X_tr, y_tr, X_va, y_va, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate_nn)
    criterion = nn.HuberLoss()  # robust to zero-inflated load
    train_loader = _seq_loader(X_tr, y_tr, shuffle=True)
    best_val, best_state, stale = float("inf"), copy.deepcopy(model.state_dict()), 0
    start = time.perf_counter()
    history = []
    for epoch in range(cfg.epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_losses = []
            for xb, yb in _seq_loader(X_va, y_va):
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_losses.append(criterion(model(xb), yb).item() * len(xb))
            val_loss = sum(val_losses) / max(sum(len(yb) for _, yb in _seq_loader(X_va, y_va)), 1)
        history.append(val_loss)
        if val_loss < best_val - 1e-6:
            best_val, best_state, stale = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= cfg.patience:
                break
    seconds = time.perf_counter() - start
    model.load_state_dict(best_state)
    return model, best_val, seconds


# Sequence targets are the standardized target channel; convert predictions back to
# log scale for a like-for-like comparison with the tabular models.
_seq_mu = float(np.nanmean(df[SEQ_TARGET_COL].to_numpy()[np.isin(np.arange(len(df)), splits["train"])]))
_seq_sd = max(float(np.nanstd(df[SEQ_TARGET_COL].to_numpy()[np.isin(np.arange(len(df)), splits["train"])])), 1e-6)
def _to_log(pred_std):
    return pred_std * _seq_sd + _seq_mu

print(f"GRU eval windows are capped at {cfg.max_eval_windows_per_building} per building "
      f"per split (cfg.max_eval_windows_per_building) and only one sequence length's "
      f"window arrays are held in memory at a time.")

gru_seed_rows = []
gru_state_dicts = {}   # (L, seed) -> small CPU state_dict (NOT the bulky window arrays)
gru_input_dim = {}     # L -> input feature dim (constant across seeds for a given L)
SEQ_FEATURE_COLS = SEQUENCE_FEATURES

for L in cfg.seq_lengths:
    mem_report(f"before building L={L} sequence windows")
    X_tr, y_tr, _ = build_sequences(df, SEQ_FEATURE_COLS, L, "train", cfg.max_train_windows_per_building, cfg.random_seed)
    X_va, y_va, _ = build_sequences(df, SEQ_FEATURE_COLS, L, "val", max_windows_per_building=cfg.max_eval_windows_per_building, seed=cfg.random_seed)
    X_te, y_te, _ = build_sequences(df, SEQ_FEATURE_COLS, L, "test", max_windows_per_building=cfg.max_eval_windows_per_building, seed=cfg.random_seed)
    if len(X_tr) == 0 or len(X_va) == 0:
        print(f"L={L}: insufficient sequence data; skipped.")
        del X_tr, y_tr, X_va, y_va, X_te, y_te
        gc.collect()
        continue
    y_va_log = _to_log(y_va)
    input_dim = X_tr.shape[2]
    gru_input_dim[L] = input_dim
    for seed in range(cfg.random_seed, cfg.random_seed + cfg.n_seeds):
        set_global_seed(seed)
        model = GRUSeqRegressor(input_dim, hidden_dim=cfg.gru_hidden)
        model, best_val, secs = _train_one(model, X_tr, y_tr, X_va, y_va, seed)
        val_log = _to_log(predict_torch(model, X_va))
        val_rmse = math.sqrt(mean_squared_error(y_va_log, val_log))
        gru_seed_rows.append({"seq_len": L, "seed": seed, "val_rmse_log": val_rmse, "seconds": secs})
        gru_state_dicts[(L, seed)] = copy.deepcopy(model.cpu().state_dict())
        print(f"GRU L={L} seed={seed}: val RMSE(log)={val_rmse:.5f}")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    # This sequence length is fully trained+evaluated: drop its (potentially large)
    # window arrays before moving on so at most one L's worth is ever resident.
    del X_tr, y_tr, X_va, y_va, X_te, y_te, y_va_log
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    mem_report(f"after finishing L={L}")

gru_seed_df = pd.DataFrame(gru_seed_rows)
if not gru_seed_df.empty:
    gru_seed_df.to_csv(cfg.output_dir / "tables" / "gru_seed_results.csv", index=False)
    # Select sequence length by MEAN validation RMSE across seeds (seed-robust choice).
    by_len = gru_seed_df.groupby("seq_len")["val_rmse_log"].mean()
    best_L = int(by_len.idxmin())
    best_seeds = gru_seed_df[gru_seed_df["seq_len"] == best_L].sort_values("val_rmse_log")
    best_seed = int(best_seeds.iloc[0]["seed"])
    mean_rmse = best_seeds["val_rmse_log"].mean()
    sd_rmse = best_seeds["val_rmse_log"].std(ddof=1)
    print(f"Selected sequence length L={best_L} (mean val RMSE {mean_rmse:.5f} \u00b1 {sd_rmse:.5f} over {cfg.n_seeds} seeds)")

    GRU_NAME = f"GRU sequence {best_L}h (best of {cfg.n_seeds} seeds)"

    # Rebuild window arrays ONLY for the winning sequence length, and reload every
    # seed's model for that length from its saved (tiny) state_dict. This reproduces
    # the same `gru_runs[(L, seed)] = {"model", "X_va", "X_te", ...}` structure the
    # rest of the notebook expects, but without ever holding more than one L's arrays
    # in memory at once.
    X_va, y_va, pos_val_global = build_sequences(df, SEQ_FEATURE_COLS, best_L, "val", max_windows_per_building=cfg.max_eval_windows_per_building, seed=cfg.random_seed)
    X_te, y_te, pos_test_global = build_sequences(df, SEQ_FEATURE_COLS, best_L, "test", max_windows_per_building=cfg.max_eval_windows_per_building, seed=cfg.random_seed)

    gru_runs = {}
    for seed in range(cfg.random_seed, cfg.random_seed + cfg.n_seeds):
        key = (best_L, seed)
        if key not in gru_state_dicts:
            continue
        m = GRUSeqRegressor(gru_input_dim[best_L], hidden_dim=cfg.gru_hidden)
        m.load_state_dict(gru_state_dicts[key])
        # X_va / X_te are shared (referenced, not copied) across every seed of best_L.
        gru_runs[key] = {"model": m, "X_te": X_te, "X_va": X_va, "y_te": y_te, "y_va": y_va}

    best_run = gru_runs[(best_L, best_seed)]
    gru_seq_model = best_run["model"]
    gru_test_log = _to_log(predict_torch(gru_seq_model, best_run["X_te"]))
    gru_val_log = _to_log(predict_torch(gru_seq_model, best_run["X_va"]))
    gru_y_test_log = _to_log(best_run["y_te"])
    gru_y_val_log = _to_log(best_run["y_va"])

    # Sequence targets cover a (capped, i.i.d. per building) subset of val/test rows.
    # Metrics are therefore computed on matched rows only, and the mask is exported so
    # downstream fusion/evaluation can align predictions to the same rows.
    val_sorted = np.sort(np.asarray(splits["val"])); test_sorted = np.sort(np.asarray(splits["test"]))
    val_ranks = np.searchsorted(val_sorted, pos_val_global)
    test_ranks = np.searchsorted(test_sorted, pos_test_global)
    assert np.array_equal(val_sorted[val_ranks], pos_val_global) and np.array_equal(test_sorted[test_ranks], pos_test_global)
    gru_val_mask = np.zeros(len(y_val), dtype=bool); gru_val_mask[val_ranks] = True
    gru_test_mask = np.zeros(len(y_test), dtype=bool); gru_test_mask[test_ranks] = True

    # Protocol A: evaluate ALL seeds at the selected L on test (mean +- sd reported),
    # not only the validation-selected best seed.
    all_seed_test_rows = []
    for (run_L, run_seed), run in sorted(gru_runs.items()):
        if run_L != best_L:
            continue
        p_log = _to_log(predict_torch(run["model"], run["X_te"]))
        m = regression_metrics(_to_log(run["y_te"]), p_log)
        all_seed_test_rows.append({"seq_len": run_L, "seed": run_seed, **m})
    gru_test_all_seeds_df = pd.DataFrame(all_seed_test_rows)
    gru_test_all_seeds_df.to_csv(cfg.output_dir / "tables" / "gru_test_all_seeds.csv", index=False)
    print("All-seed test RMSE(log) at L=%d: mean=%.5f sd=%.5f" % (
        best_L,
        gru_test_all_seeds_df["RMSE_log"].mean(),
        gru_test_all_seeds_df["RMSE_log"].std(ddof=1)))

    gru_row = {"Model": GRU_NAME,
               **regression_metrics(gru_y_test_log, gru_test_log),
               "Train_seconds": float(best_seeds.iloc[0]["seconds"])}
    results.append(gru_row)
    train_seconds[GRU_NAME] = gru_row["Train_seconds"]
    models[GRU_NAME] = gru_seq_model
    # Subset-level predictions (arrays align with gru_*_mask positions, not full y).
    predictions_val[GRU_NAME] = gru_val_log
    predictions_test[GRU_NAME] = gru_test_log
    SEQ_TEST = {"mask": gru_test_mask, "y_log": gru_y_test_log, "X": best_run["X_te"], "y_std": best_run["y_te"]}
    SEQ_VAL = {"mask": gru_val_mask, "y_log": gru_y_val_log, "X": best_run["X_va"], "y_std": best_run["y_va"]}
    _chk_v = np.abs(y_val[gru_val_mask] - gru_y_val_log)
    _chk_t = np.abs(y_test[gru_test_mask] - gru_y_test_log)
    print(f"[align-check] val: max|y_val[mask]-y_log|={_chk_v.max():.6f} over {int(gru_val_mask.sum())} matched rows; "
          f"test: max={_chk_t.max():.6f} over {int(gru_test_mask.sum())} matched rows")
    # Threshold is 1e-2, not 1e-3: the standardize -> GRU -> rescale round-trip runs
    # in float32, which alone produces ~1e-3-scale residuals on correctly aligned
    # rows. A genuine row/order misalignment (e.g. the splits-not-position-sorted
    # bug this check was added to catch) produces O(1) differences, not O(1e-3).
    if _chk_v.max() > 1e-2 or _chk_t.max() > 1e-2:
        print("[align-check] *** MISALIGNMENT: GRU window targets do not match the split targets at "
              "the mask ranks. The fusion-gate cell has its own independent tripwire and may still "
              "halt the run there; see the [tripwire-dump] output if it does. ***")
    print(GRU_NAME, {k: (round(v, 5) if isinstance(v, float) else v) for k, v in gru_row.items()})

    # We no longer need the per-length state_dicts of the non-selected lengths.
    gru_state_dicts = {k: v for k, v in gru_state_dicts.items() if k[0] == best_L}
    gc.collect()
    mem_report("after GRU sequence-length selection")
else:
    raise RuntimeError("No GRU sequence runs completed; check sequence settings.")


In [ ]:
# Training/validation convergence for the selected sequence length across seeds.
if not gru_seed_df.empty:
    # Re-plot from recorded runs: capture histories is heavy, so record final loss
    # curves for the best length during training instead.
    fig, ax = plt.subplots(figsize=(10, 6))
    sel = gru_seed_df[gru_seed_df["seq_len"] == best_L]
    ax.bar([str(s) for s in sel["seed"]], sel["val_rmse_log"], color="#4C72B0")
    ax.set_xlabel("Training seed")
    ax.set_ylabel("Validation RMSE (log scale)")
    ax.set_title(f"GRU sequence L={best_L}: validation RMSE across {cfg.n_seeds} seeds")
    plt.tight_layout()
    plt.savefig(cfg.output_dir / "figures" / "gru_seed_val_rmse.png", dpi=180, bbox_inches="tight")
    plt.savefig(cfg.output_dir / "figures" / "gru_seed_val_rmse.pdf", dpi=180, bbox_inches="tight")
    show_and_close(fig)
    display(sel.reset_index(drop=True))


## 8. Validation-only selection and fusion gate

Fusion is not automatically accepted.

Keep fusion only if:
- validation RMSE improves over LightGBM + Lag by the preregistered threshold;
- improvement is consistent across at least 4/5 GRU seeds.

Otherwise LightGBM + Lag is selected.

In [ ]:
# Validation-only fusion search on MATCHED sequence rows, with a retention gate.
# The fusion is kept only if it beats the strongest tree model on validation by at
# least cfg.fusion_min_improvement with seed-consistent sign. Otherwise the tree
# model alone is the deployed model and NO fusion row enters the results.
def strongest_tree_model():
    # Select by VALIDATION RMSE. `results` rows hold TEST metrics; selecting on them
    # would be test-set model selection (leakage).
    tree_names = [n for n in predictions_val if "GRU" not in n]
    if not tree_names:
        raise RuntimeError("No tree models registered.")
    return min(tree_names, key=lambda n: math.sqrt(mean_squared_error(y_val, predictions_val[n])))

TREE_NAME = strongest_tree_model()
print("Strongest validation tree model:", TREE_NAME)

tree_val_on_mask = predictions_val[TREE_NAME][SEQ_VAL["mask"]]
tree_test_on_mask = predictions_test[TREE_NAME][SEQ_TEST["mask"]]


def alpha_search_matched(y_true, pred_gru, pred_tree, step):
    alphas = np.round(np.arange(0.0, 1.0 + step / 2, step), 10)
    rows = []
    for alpha in alphas:
        fused = alpha * pred_gru + (1 - alpha) * pred_tree
        rows.append({"alpha_gru": float(alpha), "alpha_lgbm": float(1 - alpha),
                     PRIMARY_METRIC: math.sqrt(mean_squared_error(y_true, fused))})
    table = pd.DataFrame(rows)
    return table.loc[table[PRIMARY_METRIC].idxmin()].to_dict(), table


coarse_best, coarse_alpha_table = alpha_search_matched(
    SEQ_VAL["y_log"], predictions_val[GRU_NAME], tree_val_on_mask, step=0.05)
fine_best, fine_alpha_table = alpha_search_matched(
    SEQ_VAL["y_log"], predictions_val[GRU_NAME], tree_val_on_mask, step=0.01)
print("Best coarse-grid alpha:", coarse_best)
print("Best fine-grid alpha:", fine_best)

tree_val_rmse = math.sqrt(mean_squared_error(SEQ_VAL["y_log"], tree_val_on_mask))
# Tripwire: matched rows are a strict subset of validation rows, so the tree's
# RMSE on them cannot be far above its full-validation RMSE. A large ratio means
# predictions/targets are misaligned (stale notebook build) -- refuse to ship it.
_full_val_rmse = math.sqrt(mean_squared_error(y_val, predictions_val[TREE_NAME]))
if tree_val_rmse > max(2.0 * _full_val_rmse, _full_val_rmse + 0.05):
    _chk_v = np.abs(y_val[SEQ_VAL["mask"]] - SEQ_VAL["y_log"])
    _chk_t = np.abs(y_test[SEQ_TEST["mask"]] - SEQ_TEST["y_log"])
    _p = predictions_val[TREE_NAME][SEQ_VAL["mask"]]
    _rmse_on_yval = math.sqrt(mean_squared_error(y_val[SEQ_VAL["mask"]], _p))
    _c_ylog = float(np.corrcoef(_p, SEQ_VAL["y_log"])[0, 1])
    _c_yval = float(np.corrcoef(_p, y_val[SEQ_VAL["mask"]])[0, 1])
    _sl, _ic = np.polyfit(SEQ_VAL["y_log"], y_val[SEQ_VAL["mask"]], 1)
    _worst = np.argsort(-_chk_v)[:5]
    _ranks = np.where(SEQ_VAL["mask"])[0]
    print("[tripwire-dump] n_mask=%d n_ylog=%d len(y_val)=%d" % (int(SEQ_VAL["mask"].sum()), len(SEQ_VAL["y_log"]), len(y_val)))
    print("[tripwire-dump] splits ascending: " + ", ".join(
        "%s=%s" % (k, bool(np.all(np.asarray(splits[k])[1:] >= np.asarray(splits[k])[:-1])))
        for k in ("train", "val", "test")))
    print("[tripwire-dump] val max|y_val[mask]-y_log|=%.6f mean=%.6f | test max=%.6f" % (_chk_v.max(), _chk_v.mean(), _chk_t.max()))
    print("[tripwire-dump] tree RMSE vs y_val[mask]=%.5f | vs y_log=%.5f | full-val=%.5f" % (_rmse_on_yval, tree_val_rmse, _full_val_rmse))
    print("[tripwire-dump] corr(tree_mask_preds, y_log)=%.4f | corr(tree_mask_preds, y_val[mask])=%.4f" % (_c_ylog, _c_yval))
    print("[tripwire-dump] polyfit y_log -> y_val[mask]: slope=%.4f intercept=%.4f" % (_sl, _ic))
    print("[tripwire-dump] worst ranks:", _ranks[_worst].tolist())
    print("[tripwire-dump] y_val[mask] at worst:", np.round(y_val[SEQ_VAL["mask"]][_worst], 3).tolist())
    print("[tripwire-dump] y_log    at worst:", np.round(SEQ_VAL["y_log"][_worst], 3).tolist())
    raise RuntimeError(
        f"IMPOSSIBLE fusion-gate comparison: tree RMSE on matched validation "
        f"rows = {tree_val_rmse:.5f} but full-validation RMSE = {_full_val_rmse:.5f}. "
        f"Matched rows are a SUBSET of validation rows, so this ratio signals row "
        f"misalignment or target corruption -- see the [tripwire-dump] lines above "
        f"for the exact broken junction. Fusion is NOT retained.")
best_alpha = float(fine_best["alpha_gru"])
fused_val_rmse = float(fine_best[PRIMARY_METRIC])
relative_improvement = (tree_val_rmse - fused_val_rmse) / tree_val_rmse

# Seed-consistency requirement: at the selected alpha, the fusion must beat the
# tree model on validation for at least cfg.fusion_seed_consistency of the seeds.
seed_val_improvements = []
for (run_L, run_seed), run in gru_runs.items():
    if run_L != best_L:
        continue
    pv = _to_log(predict_torch(run["model"], run["X_va"]))
    fused_r = math.sqrt(mean_squared_error(SEQ_VAL["y_log"],
                                           best_alpha * pv + (1 - best_alpha) * tree_val_on_mask))
    seed_val_improvements.append((tree_val_rmse - fused_r) / tree_val_rmse)
seed_consistency = float(np.mean([imp > 0 for imp in seed_val_improvements])) if seed_val_improvements else 0.0
print(f"Seed consistency at alpha={best_alpha}: {seed_val_improvements} -> "
      f"{100*seed_consistency:.0f}% of seeds improve (requires >= {100*cfg.fusion_seed_consistency:.0f}%)")

gate_pass = ((best_alpha > 0.0)
             and (relative_improvement >= cfg.fusion_min_improvement)
             and (seed_consistency >= cfg.fusion_seed_consistency))
print(f"Validation tree RMSE={tree_val_rmse:.5f}, fused RMSE={fused_val_rmse:.5f}, "
      f"relative improvement={100*relative_improvement:.2f}% (gate requires >= {100*cfg.fusion_min_improvement:.1f}% with alpha>0)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fine_alpha_table["alpha_gru"], fine_alpha_table[PRIMARY_METRIC])
ax.axvline(best_alpha, linestyle=":", label=f"Validation optimum={best_alpha:.2f}")
# (stale "Paper alpha=0.38" reference line removed 2026-09-19: the paper
# reports validation optimum alpha=0.06 and deploys alpha=0; no alpha=0.38
# appears anywhere in the manuscript.)
ax.set_xlabel("GRU weight α")
ax.set_ylabel("Validation RMSE (log scale, matched rows)")
ax.set_title("Ensemble Weight Sensitivity (validation only)")
ax.legend()
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "alpha_sensitivity.png", dpi=180, bbox_inches="tight")
plt.savefig(cfg.output_dir / "figures" / "alpha_sensitivity.pdf", dpi=180, bbox_inches="tight")
show_and_close(fig)

FUSION_RETAINED = bool(gate_pass)
FINAL_ALPHA = best_alpha if FUSION_RETAINED else 0.0
print("Fusion retained:", FUSION_RETAINED)
mem_report("after validation-only fusion search")


In [ ]:
# Register fusion rows ONLY when the validation gate retained the GRU path.
if FUSION_RETAINED:
    def fused_predictions(alpha: float, split: str = "test") -> np.ndarray:
        if split == "test":
            return alpha * predictions_test[GRU_NAME] + (1 - alpha) * predictions_test[TREE_NAME][SEQ_TEST["mask"]]
        return alpha * predictions_val[GRU_NAME] + (1 - alpha) * predictions_val[TREE_NAME][SEQ_VAL["mask"]]

    ENSEMBLE_NAME = f"TropiLite-IE (fused α={FINAL_ALPHA:.2f}, validated)"
    fused_row = {"Model": ENSEMBLE_NAME,
                 **regression_metrics(SEQ_TEST["y_log"], fused_predictions(FINAL_ALPHA, "test")),
                 "Train_seconds": train_seconds[TREE_NAME] + train_seconds[GRU_NAME]}
    results.append(fused_row)
    models[ENSEMBLE_NAME] = {"Tree": TREE_NAME, "GRU": GRU_NAME, "alpha": FINAL_ALPHA}
    train_seconds[ENSEMBLE_NAME] = fused_row["Train_seconds"]
    predictions_val[ENSEMBLE_NAME] = fused_predictions(FINAL_ALPHA, "val")
    predictions_test[ENSEMBLE_NAME] = fused_predictions(FINAL_ALPHA, "test")
    print(ENSEMBLE_NAME, {k: (round(v, 5) if isinstance(v, float) else v) for k, v in fused_row.items()})
else:
    ENSEMBLE_NAME = TREE_NAME
    print(f"Gate not passed: the deployed model is '{TREE_NAME}' alone (no fusion row registered).")

FINAL_ENSEMBLE_NAME = ENSEMBLE_NAME
print("Final deployed model:", FINAL_ENSEMBLE_NAME)


## 9. Magnitude pruning and dynamic 8-bit quantization


In [ ]:
# Compression is applied only when the GRU is actually part of the deployed pipeline.
# A zero-weight or non-selected GRU is NOT executed or reported (per the protocol).
quantized_gru = None
pruned_gru = None
if FUSION_RETAINED and FINAL_ALPHA > 0.0:
    def apply_global_magnitude_pruning(model: nn.Module, amount: float = 0.40) -> nn.Module:
        pruned = copy.deepcopy(model).cpu().eval()
        parameters_to_prune = []
        for module in pruned.modules():
            for name, param in list(module.named_parameters(recurse=False)):
                if "weight" in name and param.ndim >= 2:
                    parameters_to_prune.append((module, name))
        prune.global_unstructured(parameters_to_prune, pruning_method=prune.L1Unstructured, amount=amount)
        for module, name in parameters_to_prune:
            prune.remove(module, name)
        return pruned

    def dynamic_quantize_model(model: nn.Module) -> nn.Module:
        quantize_fn = getattr(torch.ao.quantization, "quantize_dynamic", None)
        if quantize_fn is None:
            raise RuntimeError("Dynamic quantization is unavailable in this PyTorch build")
        return quantize_fn(model.cpu().eval(), {nn.Linear, nn.GRU}, dtype=torch.qint8)

    pruned_gru = apply_global_magnitude_pruning(gru_seq_model, cfg.pruning_amount)
    try:
        quantized_gru = dynamic_quantize_model(pruned_gru)
        q_val_gru = _to_log(predict_torch(quantized_gru, gru_runs[(best_L, best_seed)]["X_va"]))
        q_test_gru = _to_log(predict_torch(quantized_gru, gru_runs[(best_L, best_seed)]["X_te"]))
        fused_row = {"Model": "TropiLite-IE (pruned + dynamic int8 GRU)",
                     **regression_metrics(SEQ_TEST["y_log"],
                                          FINAL_ALPHA * q_test_gru + (1 - FINAL_ALPHA) * predictions_test[TREE_NAME][SEQ_TEST["mask"]]),
                     "Train_seconds": np.nan}
        results.append(fused_row)
        models["TropiLite-IE (pruned + dynamic int8 GRU)"] = {"Tree": TREE_NAME, "Quantized_GRU": quantized_gru, "alpha": FINAL_ALPHA}
        predictions_val["TropiLite-IE (pruned + dynamic int8 GRU)"] = FINAL_ALPHA * q_val_gru + (1 - FINAL_ALPHA) * predictions_val[TREE_NAME][SEQ_VAL["mask"]]
        predictions_test["TropiLite-IE (pruned + dynamic int8 GRU)"] = FINAL_ALPHA * q_test_gru + (1 - FINAL_ALPHA) * predictions_test[TREE_NAME][SEQ_TEST["mask"]]
        print("Compressed-GRU fusion row registered:", {k: (round(v, 5) if isinstance(v, float) else v) for k, v in fused_row.items()})
    except Exception as exc:
        quantized_gru = None
        warnings.warn(f"Dynamic quantization failed on this platform: {exc}")
else:
    print("GRU is not part of the deployed pipeline (fusion gate not passed); "
          "compression and quantized-pipeline reporting are skipped by design.")
gc.collect()
mem_report("after pruning + dynamic quantization")


## 10. Global and tropical-subset results


In [ ]:
results_df = pd.DataFrame(results).drop_duplicates("Model", keep="last").sort_values("RMSE_log").reset_index(drop=True)
display(results_df)
results_df.to_csv(cfg.output_dir / "tables" / "global_results.csv", index=False)


In [ ]:
def bootstrap_rmse_by_building(y_true, y_pred, building_ids, iterations=1000, seed=42):
    frame = pd.DataFrame({"building_id": building_ids, "sqerr": (y_true - y_pred) ** 2})
    group = frame.groupby("building_id")["sqerr"].agg(["sum", "count"])
    ids = group.index.to_numpy()
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(iterations):
        sampled = rng.choice(ids, size=len(ids), replace=True)
        rows = group.loc[sampled]
        boot.append(math.sqrt(rows["sum"].sum() / rows["count"].sum()))
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def building_level_loss_test(y_true, pred_model, pred_reference, building_ids):
    """Paired per-building squared-loss-difference t-test (building = unit)."""
    frame = pd.DataFrame({
        "building_id": building_ids,
        "loss_model": (y_true - pred_model) ** 2,
        "loss_ref": (y_true - pred_reference) ** 2,
    })
    by_building = frame.groupby("building_id")[["loss_model", "loss_ref"]].mean()
    d = by_building["loss_model"] - by_building["loss_ref"]
    if len(d) < 2 or d.std(ddof=1) == 0:
        return np.nan, np.nan
    stat = d.mean() / (d.std(ddof=1) / math.sqrt(len(d)))
    p = 2 * stats.t.sf(abs(stat), df=len(d) - 1)
    return float(stat), float(p)


def holm_correct(pvals):
    """Holm-Bonferroni step-down adjustment."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(np.where(np.isnan(p), np.inf, p))
    m = np.sum(~np.isnan(p))
    adjusted = np.full_like(p, np.nan)
    running = 0.0
    for rank, idx in enumerate(order):
        if np.isnan(p[idx]):
            continue
        adj = min(1.0, (m - rank) * p[idx])
        running = max(running, adj)
        adjusted[idx] = running
    return adjusted

eligible = [(name, pred) for name, pred in predictions_test.items() if len(pred) == len(y_test)]
print(f"Computing bootstrap CIs for {len(eligible)} full-coverage models "
      f"({cfg.bootstrap_iterations} building-level resamples each)...")

ci_rows = []
b_ids_full = test_meta["building_id"].to_numpy()
for name, pred in eligible:
    low, high = bootstrap_rmse_by_building(y_test, pred, b_ids_full,
                                           iterations=cfg.bootstrap_iterations, seed=cfg.random_seed)
    ci_rows.append({"Model": name, "RMSE_log": math.sqrt(mean_squared_error(y_test, pred)),
                    "CI_95_low": low, "CI_95_high": high, "Coverage": "full test rows"})
# Subset-coverage models (GRU / retained fusion) get matched-row CIs, flagged as such.
for name, pred in predictions_test.items():
    if len(pred) == len(y_test):
        continue
    m = SEQ_TEST["mask"]
    # Check if the lengths match for subset models before calling bootstrap_rmse_by_building
    if len(SEQ_TEST["y_log"]) == len(pred):
        low, high = bootstrap_rmse_by_building(SEQ_TEST["y_log"], pred, b_ids_full[m],
                                               iterations=cfg.bootstrap_iterations, seed=cfg.random_seed)
        ci_rows.append({"Model": name, "RMSE_log": math.sqrt(mean_squared_error(SEQ_TEST["y_log"], pred)),
                        "CI_95_low": low, "CI_95_high": high,
                        "Coverage": f"matched rows only (n={int(m.sum())})"})
    else:
        print(f"Warning: Skipping CI calculation for {name} due to mismatched prediction and target lengths for subset coverage. Expected {len(SEQ_TEST['y_log'])}, got {len(pred)}.")


if ci_rows:
    ci_df = pd.DataFrame(ci_rows).sort_values("RMSE_log").reset_index(drop=True)
else:
    print("Warning: ci_rows is empty, creating an empty DataFrame with expected columns.")
    ci_df = pd.DataFrame(columns=["Model", "RMSE_log", "CI_95_low", "CI_95_high", "Coverage"])
display(ci_df)
ci_df.to_csv(cfg.output_dir / "tables" / "bootstrap_ci_by_building.csv", index=False)


def comparison_arrays(name_a: str, name_b: str):
    """Return (y, pred_a, pred_b, building_ids) on the common evaluation rows.

    If either model covers only the matched sequence rows, both are restricted to
    those rows so the comparison stays paired and like-for-like.
    """
    is_a_subset = (len(predictions_test[name_a]) != len(y_test))
    is_b_subset = (len(predictions_test[name_b]) != len(y_test))

    if is_a_subset or is_b_subset:
        m = SEQ_TEST["mask"]
        y_c = SEQ_TEST["y_log"]
        pred_a = predictions_test[name_a] if is_a_subset else predictions_test[name_a][m]
        pred_b = predictions_test[name_b] if is_b_subset else predictions_test[name_b][m]
        building_ids = test_meta["building_id"].to_numpy()[m]
        return (y_c, pred_a, pred_b, building_ids)
    else:
        return (y_test, predictions_test[name_a], predictions_test[name_b], b_ids_full)


# Pre-registered primary comparisons + all other pairs (exploratory, Holm-corrected).
PRIMARY_COMPARISONS = [
    ("LightGBM + Lag Features", "LightGBM (No Enthalpy)"),
    ("LightGBM + Enthalpy", "LightGBM (No Enthalpy)"),
    (GRU_NAME, "LightGBM + Lag Features"),
]
if FUSION_RETAINED:
    PRIMARY_COMPARISONS.append((FINAL_ENSEMBLE_NAME, TREE_NAME))

def comparison_exists(a, b, rows):
    return any({a, b} == set(r["Comparison"].split(" vs ")) for r in rows)

pair_rows = []
for model_name, ref_name in PRIMARY_COMPARISONS:
    if model_name in predictions_test and ref_name in predictions_test:
        y_c, pa, pb, ids = comparison_arrays(model_name, ref_name)
        # Added check for lengths before calling building_level_loss_test
        if len(y_c) == len(pa) and len(y_c) == len(pb) and len(y_c) == len(ids):
            stat, p = building_level_loss_test(y_c, pa, pb, ids)
            pair_rows.append({"Comparison": f"{model_name} vs {ref_name}", "Type": "primary",
                              "t_stat": stat, "p_value": p})
        else:
            print(f"Warning: Skipping comparison {model_name} vs {ref_name} due to mismatched lengths.")
for i, name_a in enumerate(predictions_test):
    for name_b in list(predictions_test)[i + 1:]:
        if comparison_exists(name_a, name_b, pair_rows):
            continue
        y_c, pa, pb, ids = comparison_arrays(name_a, name_b)
        # Added check for lengths before calling building_level_loss_test
        if len(y_c) == len(pa) and len(y_c) == len(pb) and len(y_c) == len(ids):
            stat, p = building_level_loss_test(y_c, pa, pb, ids)
            pair_rows.append({"Comparison": f"{name_a} vs {name_b}", "Type": "exploratory",
                              "t_stat": stat, "p_value": p})
        else:
            print(f"Warning: Skipping comparison {name_a} vs {name_b} due to mismatched lengths.")


if pair_rows:
    pair_df = pd.DataFrame(pair_rows)
    mask_primary = pair_df["Type"] == "primary"
    pair_df["p_holm"] = np.nan
    pair_df.loc[mask_primary, "p_holm"] = holm_correct(pair_df.loc[mask_primary, "p_value"].to_numpy())
    exploratory_mask = ~mask_primary
    pair_df.loc[exploratory_mask, "p_holm"] = holm_correct(pair_df.loc[exploratory_mask, "p_value"].to_numpy())
    pair_df = pair_df.sort_values(["Type", "p_value"]).reset_index(drop=True)
else:
    print("Warning: pair_rows is empty, creating an empty DataFrame with expected columns.")
    pair_df = pd.DataFrame(columns=["Comparison", "Type", "t_stat", "p_value", "p_holm"])
display(pair_df)
pair_df.to_csv(cfg.output_dir / "tables" / "paired_model_tests.csv", index=False)


# --- Moving-block bootstrap on test residuals (design doc section 4):
# blocks of cfg.block_length_hours within each building's ordered test stream;
# blocks are resampled with replacement and the pooled RMSE recorded.
def block_bootstrap_rmse(y_true, y_pred, building_ids, timestamps, block_hours=168, iterations=1000, seed=42):
    frame = pd.DataFrame({"b": building_ids, "ts": pd.to_datetime(timestamps), "sq": (y_true - y_pred) ** 2})
    frame = frame.sort_values(["b", "ts"])
    blocks = []
    for _, g in frame.groupby("b"):
        sq = g["sq"].to_numpy()
        for s in range(0, len(sq), block_hours):
            blocks.append(sq[s:s + block_hours])
    n_blocks = len(blocks)
    block_means = np.array([b.mean() for b in blocks])
    block_lens = np.array([len(b) for b in blocks])
    rng = np.random.default_rng(seed)
    out = np.empty(iterations)
    for it in range(iterations):
        idx = rng.integers(0, n_blocks, n_blocks)
        num = float(np.sum(block_means[idx] * block_lens[idx]))
        den = float(block_lens[idx].sum())
        out[it] = math.sqrt(num / den)
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


mb_rows = []
_ts_test = test_meta["timestamp"].to_numpy()
for name, pred in eligible:
    low, high = block_bootstrap_rmse(y_test, pred, b_ids_full, _ts_test,
                                  block_hours=cfg.block_length_hours,
                                  iterations=cfg.bootstrap_iterations, seed=cfg.random_seed)
    mb_rows.append({"Model": name, "RMSE_log": math.sqrt(mean_squared_error(y_test, pred)),
                    "MB_CI_95_low": low, "MB_CI_95_high": high})

# Added if-else block to handle empty mb_rows
if mb_rows:
    mb_df = pd.DataFrame(mb_rows).sort_values("RMSE_log").reset_index(drop=True)
else:
    print("Warning: mb_rows is empty, creating an empty DataFrame with expected columns.")
    mb_df = pd.DataFrame(columns=["Model", "RMSE_log", "MB_CI_95_low", "MB_CI_95_high"])
mb_df.to_csv(cfg.output_dir / "tables" / "block_bootstrap_ci.csv", index=False)
print("Moving-block bootstrap CIs (168h blocks) written to block_bootstrap_ci.csv")
mem_report("after bootstrap + paired statistical tests")

In [ ]:
import warnings

# Define tropical_mask here
tropical_mask = test_meta["is_tropical"].to_numpy().astype(bool)

# Tropical performance-gap table. Per-building RMSE (log scale) for tropical vs
# non-tropical test buildings (Welch test). GRU is evaluated on its matched
# rows only (those with >= L hours of history), so its numbers are not directly
# comparable to the full-coverage models and are flagged as such.
def per_building_rmse(y_true, y_pred, building_ids):
    frame = pd.DataFrame({"building_id": building_ids, "sqerr": (y_true - y_pred) ** 2})
    return frame.groupby("building_id")["sqerr"].mean().pow(0.5)

gap_rows = []

# Iterate through all models in predictions_test
for name, pred in predictions_test.items():
    # Determine if the current model's predictions cover a subset of the test data
    is_subset_model = (len(pred) != len(y_test))

    if is_subset_model:
        # For subset models (GRU and fused models), use the masked test data
        m = SEQ_TEST["mask"]
        current_y_test = SEQ_TEST["y_log"]
        current_pred = pred # pred is already the subset prediction
        current_building_ids = test_meta["building_id"].to_numpy()[m]
        current_tropical_mask = tropical_mask[m]
        current_coverage_label = f"matched rows only (n={int(m.sum())})"
    else:
        # For full coverage models, use the entire test data
        current_y_test = y_test
        current_pred = pred
        current_building_ids = test_meta["building_id"].to_numpy()
        current_tropical_mask = tropical_mask
        current_coverage_label = "full test rows"

    # Ensure y_true and y_pred have consistent lengths for regression_metrics
    if len(current_y_test) != len(current_pred):
        warnings.warn(f"Skipping gap analysis for '{name}' due to inconsistent lengths: y_true={len(current_y_test)}, y_pred={len(current_pred)}")
        continue

    global_rmse = regression_metrics(current_y_test, current_pred)["RMSE_log"]

    tropical_rmse = np.nan
    degradation = np.nan
    p_value = np.nan

    # Perform tropical vs non-tropical analysis if tropical buildings exist in the current subset
    if current_tropical_mask.sum() > 0:
        tropical_rmse = regression_metrics(current_y_test[current_tropical_mask], current_pred[current_tropical_mask])["RMSE_log"]
        degradation = 100 * (tropical_rmse / global_rmse - 1)

        b_rmse = per_building_rmse(current_y_test, current_pred, current_building_ids)
        # Determine if each building in the current subset is tropical
        building_is_tropical_series = pd.Series(current_tropical_mask, index=current_building_ids)
        building_tropical_in_subset = building_is_tropical_series.groupby(level=0).max().astype(bool)

        trop_values = b_rmse[building_tropical_in_subset.reindex(b_rmse.index).fillna(False)]
        other_values = b_rmse[~building_tropical_in_subset.reindex(b_rmse.index).fillna(False)]

        if len(trop_values) > 1 and len(other_values) > 1:
            p_value = stats.ttest_ind(trop_values, other_values, equal_var=False, nan_policy="omit").pvalue
        else:
            p_value = np.nan # Not enough data for t-test

    gap_rows.append({"Model": name, "Global_RMSE_log": global_rmse, "Tropical_RMSE_log": tropical_rmse,
                     "Degradation_percent": degradation,
                     "Welch_p_tropical_vs_other_buildings": p_value, "Coverage": current_coverage_label})

gap_df = pd.DataFrame(gap_rows)
display(gap_df)
gap_df.to_csv(cfg.output_dir / "tables" / "tropical_performance_gap.csv", index=False)


# --- Pre-registered H2 decision rule (Protocol A design, section 4):
# blocks of cfg.block_length_hours within each building's ordered test stream;
# blocks are resampled with replacement and the pooled RMSE recorded.
def h2_tropical_delta_ci(iterations=None, seed=None):
    enthalpy_name = "LightGBM + Enthalpy"
    base_name = "LightGBM (No Enthalpy)"
    if enthalpy_name not in predictions_test or base_name not in predictions_test:
        return None

    # Ensure these are full coverage models, otherwise adjust y_test/predictions
    if len(predictions_test[enthalpy_name]) != len(y_test) or len(predictions_test[base_name]) != len(y_test):
        warnings.warn(f"H2 tropical delta CI requires full coverage models for '{enthalpy_name}' and '{base_name}', but one or both are subset models. Skipping H2 analysis.")
        return None

    if tropical_mask.sum() == 0:
        print("H2 not testable: no tropical test buildings in the sample.")
        return None
    b_rmse_e = per_building_rmse(y_test, predictions_test[enthalpy_name], test_meta["building_id"].to_numpy())
    b_rmse_b = per_building_rmse(y_test, predictions_test[base_name], test_meta["building_id"].to_numpy())
    building_tropical = test_meta.groupby("building_id")["is_tropical"].max().astype(bool)
    trop_ids = building_tropical[building_tropical].index
    trop_ids = trop_ids.intersection(b_rmse_e.index)
    if len(trop_ids) < 2:
        print("H2 not testable: fewer than two tropical test buildings.")
        return None
    delta = (b_rmse_e.loc[trop_ids] - b_rmse_b.loc[trop_ids]).to_numpy()
    rng = np.random.default_rng(seed if seed is not None else cfg.random_seed)
    n_iter = iterations if iterations is not None else cfg.bootstrap_iterations
    boots = []
    for _ in range(n_iter):
        s = rng.integers(0, len(delta), len(delta))
        boots.append(float(np.mean(delta[s])))
    low, high = np.percentile(boots, [2.5, 97.5])
    out = pd.DataFrame(
        [{
         "Comparison": f"{enthalpy_name} - {base_name} (per-building RMSE delta)",
         "Subset": "tropical test buildings",
         "n_buildings": int(len(trop_ids)),
         "Mean_delta": float(delta.mean()),
         "CI_95_low": float(low), "CI_95_high": float(high),
         "H2_decision": "rejected (CI excludes zero)" if (low > 0 or high < 0) else "not rejected (CI includes zero)",
        }])
    out.to_csv(cfg.output_dir / "tables" / "h2_tropical_decision.csv", index=False)
    display(out)
    return out

h2_result = h2_tropical_delta_ci()


# --- Zero-inflation diagnostic (Protocol A design, section 5).
test_zero_rate = float((y_test <= np.log1p(1e-9)).mean())
print(f"Test zero-rate (raw scale): {100*test_zero_rate:.2f}%")
pd.DataFrame([{"split": "test", "zero_rate": test_zero_rate}]).to_csv(
    cfg.output_dir / "tables" / "zero_inflation_check.csv", index=False)
if test_zero_rate > 0.20:
    warnings.warn(
        "Test zero-rate exceeds 20%; the protocol requires a two-part model "
        "(zero classifier + positive regressor) comparison before final reporting.")
else:
    print("Zero-rate below 20%; single-stage models per protocol.")

## 11. Prediction scatter and residual diagnostics


In [ ]:
_FULL_NAME = FINAL_ENSEMBLE_NAME
if len(predictions_test[FINAL_ENSEMBLE_NAME]) != len(y_test):
    _FULL_NAME = TREE_NAME
    print(f"Deployed model covers matched rows only; full-coverage diagnostics use '{TREE_NAME}'.")

plot_candidates = [
    TREE_NAME, GRU_NAME, FINAL_ENSEMBLE_NAME, "XGBoost", "CatBoost",
]
plot_models = [n for n in dict.fromkeys(plot_candidates) if n in predictions_test]

for name in plot_models:
    if len(predictions_test[name]) != len(y_test):
        m = SEQ_TEST["mask"]
        y_plot, pred_plot = SEQ_TEST["y_log"], predictions_test[name]
    else:
        y_plot, pred_plot = y_test, predictions_test[name]
    sample_ix = np.random.default_rng(cfg.random_seed).choice(len(y_plot), size=min(4000, len(y_plot)), replace=False)
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(y_plot[sample_ix], pred_plot[sample_ix], alpha=0.25, s=14)
    low = min(y_plot[sample_ix].min(), pred_plot[sample_ix].min())
    high = max(y_plot[sample_ix].max(), pred_plot[sample_ix].max())
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("Actual log meter reading")
    ax.set_ylabel("Predicted log meter reading")
    ax.set_title(f"Predicted vs Actual: {name}")
    plt.tight_layout()
    safe = "".join(ch if ch.isalnum() else "_" for ch in name)
    plt.savefig(cfg.output_dir / "figures" / f"scatter_{safe}.png", dpi=180, bbox_inches="tight")
    plt.savefig(cfg.output_dir / "figures" / f"scatter_{safe}.pdf", dpi=180, bbox_inches="tight")
    show_and_close(fig)

final_pred = predictions_test[_FULL_NAME]
residuals = y_test - final_pred
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(final_pred, residuals, alpha=0.25, s=14)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Predicted log meter reading")
ax.set_ylabel("Residual")
ax.set_title(f"Residuals: {_FULL_NAME}")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "final_residuals.png", dpi=180, bbox_inches="tight")
plt.savefig(cfg.output_dir / "figures" / "final_residuals.pdf", dpi=180, bbox_inches="tight")
show_and_close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(residuals, bins=50, alpha=0.8)
if len(np.unique(residuals)) > 1:
    kde = stats.gaussian_kde(residuals)
    grid = np.linspace(residuals.min(), residuals.max(), 300)
    bin_width = (residuals.max() - residuals.min()) / 50
    ax.plot(grid, kde(grid) * len(residuals) * bin_width)
ax.set_title(f"Residual Distribution: {_FULL_NAME}")
plt.tight_layout()
plt.savefig(cfg.output_dir / "figures" / "final_residual_distribution.png", dpi=180, bbox_inches="tight")
plt.savefig(cfg.output_dir / "figures" / "final_residual_distribution.pdf", dpi=180, bbox_inches="tight")
show_and_close(fig)


## 12. Tree SHAP explainability


In [ ]:
shap_importance_df = pd.DataFrame()
if cfg.run_shap and shap is not None:
    rng = np.random.default_rng(cfg.random_seed)
    n_shap = min(cfg.shap_sample_size, len(y_test))
    shap_ix = rng.choice(len(y_test), size=n_shap, replace=False)
    X_shap = matrices["enthalpy"].X_test[shap_ix]
    explainer = shap.TreeExplainer(lgb_enthalpy)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_importance_df = pd.DataFrame({
        "Feature": matrices["enthalpy"].feature_names,
        "Mean_abs_SHAP": np.abs(shap_values).mean(axis=0),
    }).sort_values("Mean_abs_SHAP", ascending=False)
    display(shap_importance_df.head(20))
    shap_importance_df.to_csv(cfg.output_dir / "tables" / "shap_importance.csv", index=False)

    shap.summary_plot(shap_values, X_shap, feature_names=matrices["enthalpy"].feature_names, show=False)
    plt.title("Tree SHAP Summary: LightGBM + Enthalpy")
    plt.tight_layout()
    plt.savefig(cfg.output_dir / "figures" / "shap_summary.png", dpi=180, bbox_inches="tight")
    plt.savefig(cfg.output_dir / "figures" / "shap_summary.pdf", dpi=180, bbox_inches="tight")
    show_and_close()
    del X_shap, shap_values, explainer
    gc.collect()
else:
    print("SHAP skipped. Set TROPILITE_RUN_SHAP=1 and install shap to enable it.")
mem_report("after SHAP explainability")


## 13. Artifact export, model size, and local latency


In [ ]:
model_dir = cfg.output_dir / "models"

# Export preprocessing and feature metadata.
for key, bundle in matrices.items():
    joblib.dump(bundle.imputer, model_dir / f"imputer_{key}.joblib")
    joblib.dump(bundle.scaler, model_dir / f"scaler_{key}.joblib")

with open(model_dir / "feature_sets.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets, f, indent=2)
with open(model_dir / "primary_use_mapping.json", "w", encoding="utf-8") as f:
    json.dump(primary_use_mapping, f, indent=2)
with open(model_dir / "config.json", "w", encoding="utf-8") as f:
    serializable_cfg = {k: (list(v) if isinstance(v, tuple) else str(v) if isinstance(v, Path) else v)
                        for k, v in asdict(cfg).items()}
    serializable_cfg["final_model"] = FINAL_ENSEMBLE_NAME
    serializable_cfg["fusion_retained"] = bool(FUSION_RETAINED)
    serializable_cfg["final_alpha"] = FINAL_ALPHA
    serializable_cfg["gru_seq_len"] = int(best_L) if FUSION_RETAINED or 'best_L' in dir() else None
    serializable_cfg["n_seeds"] = cfg.n_seeds
    json.dump(serializable_cfg, f, indent=2)

# Tree exports (the selected LightGBM variants).
lgb_enthalpy.booster_.save_model(str(model_dir / "lightgbm_enthalpy.txt"))
joblib.dump(lgb_enthalpy, model_dir / "lightgbm_enthalpy.joblib")
np.save(model_dir / "sample_enthalpy_input.npy", matrices["enthalpy"].X_test[:1])

# Neural exports only when the GRU survived the selection gate.
if FUSION_RETAINED:
    torch.save(gru_seq_model.cpu().state_dict(), model_dir / "gru_sequence_state_dict.pt")
    if pruned_gru is not None:
        torch.save(pruned_gru.state_dict(), model_dir / "gru_pruned_state_dict.pt")
    if quantized_gru is not None:
        torch.save(quantized_gru, model_dir / "gru_pruned_dynamic_int8_full.pt")
else:
    print("No GRU artifacts exported: the GRU was not retained by the validation gate.")


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024**2)

size_table = pd.DataFrame([
    {"Artifact": p.name, "Size_MB": file_size_mb(p)}
    for p in sorted(model_dir.glob("*")) if p.is_file()
]).sort_values("Size_MB", ascending=False)
display(size_table)
size_table.to_csv(cfg.output_dir / "tables" / "artifact_sizes.csv", index=False)


In [ ]:
def benchmark_callable(fn, x_one: np.ndarray, warmup: int = 20, repeats: int = 200) -> Dict[str, float]:
    for _ in range(warmup):
        fn(x_one)
    times_ms = []
    process = psutil.Process(os.getpid())
    before_rss = process.memory_info().rss / (1024**2)
    for _ in range(repeats):
        start = time.perf_counter_ns()
        fn(x_one)
        times_ms.append((time.perf_counter_ns() - start) / 1e6)
    after_rss = process.memory_info().rss / (1024**2)
    return {
        "Mean_latency_ms": float(np.mean(times_ms)),
        "Median_latency_ms": float(np.median(times_ms)),
        "P95_latency_ms": float(np.percentile(times_ms, 95)),
        "Predictions_per_second_from_mean": float(1000 / max(np.mean(times_ms), 1e-9)),
        "Process_RSS_delta_MB": float(after_rss - before_rss),
    }


def _matrix_key_for(name: str) -> str:
    if "No Enthalpy" in name: return "base"
    if "Enthalpy" in name and "Lag" in name: return "enthalpy_lag"
    if "Lag" in name: return "lag"
    if "Enthalpy" in name: return "enthalpy"
    return "base"

bench_rows = []
_tree_model = models[TREE_NAME]
_one_tree = matrices[_matrix_key_for(TREE_NAME)].X_test[:1]
bench_rows.append({"Component": f"Tree pathway ('{TREE_NAME}')", **benchmark_callable(lambda x: _tree_model.predict(x), _one_tree)})

if FUSION_RETAINED:
    one_seq = SEQ_TEST["X"][:1]
    repeats_seq = 100
    bench_rows.append({"Component": f"GRU sequence (L={best_L})",
                       **benchmark_callable(lambda x: predict_torch(gru_seq_model.cpu(), x), one_seq, repeats=repeats_seq)})
    # Fusion benchmark uses the deployed tree model on ITS OWN feature matrix,
    # and the GRU on sequence windows (different inputs, reported separately).
    bench_rows.append({
        "Component": f"Deployed ensemble α={FINAL_ALPHA:.2f} (tree + GRU, inputs as above)",
        **benchmark_callable(lambda x: predict_torch(gru_seq_model.cpu(), x), one_seq, repeats=repeats_seq)})
    if quantized_gru is not None:
        bench_rows.append({
            "Component": "Quantized GRU",
            **benchmark_callable(lambda x: predict_torch(quantized_gru, x), one_seq, repeats=repeats_seq)})
else:
    print(f"Only the tree pathway is benchmarked: the deployed model is '{TREE_NAME}' alone.")

benchmark_df = pd.DataFrame(bench_rows)
display(benchmark_df)
benchmark_df.to_csv(cfg.output_dir / "tables" / "local_latency.csv", index=False)
print("These are local-machine measurements, not Raspberry Pi measurements. "
      "H4 (edge feasibility) remains unvalidated until Raspberry Pi benchmarks are run.")


## 14. HVAC pre-cooling schedule and modeled scenario projection


In [ ]:
_FULL_NAME = FINAL_ENSEMBLE_NAME
if len(predictions_test[FINAL_ENSEMBLE_NAME]) != len(y_test):
    _FULL_NAME = TREE_NAME
    print(f"Deployed model covers matched rows only; full-coverage diagnostics use '{TREE_NAME}'.")

def build_precooling_schedule(
    prediction_log: np.ndarray,
    metadata: pd.DataFrame,
    percentile: float = 0.75,
    lead_minutes: int = 30,
    setpoint_reduction_c: float = 1.0,
) -> pd.DataFrame:
    """Flag predicted peaks and create pre-cooling actions.

    This creates the rule-based schedule described by the paper. It does not calculate
    energy savings because the paper does not supply a complete building-specific
    thermodynamic coefficient for converting each event into kWh.
    """
    schedule = metadata.copy()
    schedule["prediction_log"] = prediction_log
    schedule["prediction_raw"] = np.expm1(np.maximum(prediction_log, 0))
    threshold = schedule.groupby("building_id")["prediction_raw"].transform(lambda s: s.quantile(percentile))
    schedule["predicted_peak"] = schedule["prediction_raw"] > threshold
    schedule["precool_start"] = schedule["timestamp"] - pd.to_timedelta(lead_minutes, unit="m")
    schedule["setpoint_reduction_c"] = np.where(schedule["predicted_peak"], setpoint_reduction_c, 0.0)
    return schedule.sort_values(["building_id", "timestamp"])


schedule = build_precooling_schedule(predictions_test[_FULL_NAME], test_meta)
display(schedule[schedule["predicted_peak"]].head(20))
schedule.to_csv(cfg.output_dir / "tables" / "precooling_schedule.csv", index=False)
print("Peak events:", int(schedule["predicted_peak"].sum()))


In [ ]:
def paper_hvac_projection(
    floor_area_sqft: float = 5000,
    high_intensity_kwh_sqft_year: float = 90,
    tariff_bdt_kwh: float = 70,
    scenario_a_savings_fraction: float = 0.056,
    scenario_b_savings_fraction: float = 0.090,
) -> pd.DataFrame:
    """User-specified ILLUSTRATIVE scenario projection.

    The savings fractions below are scenario INPUTS, not empirical estimates and not
    calculated from the test-set prediction errors above. No measured or simulated
    HVAC energy result exists in this notebook; H5 remains unsupported until a
    calibrated simulation or field experiment is performed.
    """
    baseline = floor_area_sqft * high_intensity_kwh_sqft_year
    scenarios = [
        ("Reactive baseline (no pre-cooling)", 0.0),
        (f"Pre-cooling scenario A ({scenario_a_savings_fraction*100:.1f}% assumed saving)", scenario_a_savings_fraction),
        (f"Pre-cooling scenario B ({scenario_b_savings_fraction*100:.1f}% assumed saving)", scenario_b_savings_fraction),
    ]
    rows = []
    for name, saving in scenarios:
        energy = baseline * (1 - saving)
        rows.append({
            "Scenario": name,
            "Annual_Energy_kWh": energy,
            "Annual_Cost_BDT": energy * tariff_bdt_kwh,
            "Assumed_Saving_percent_input": saving * 100,
            "Projected_Cost_Saving_BDT": baseline * saving * tariff_bdt_kwh,
            "Modeled_Scenario_Not_Empirical": True,
        })
    return pd.DataFrame(rows)


hvac_projection_df = paper_hvac_projection()
display(hvac_projection_df)
hvac_projection_df.to_csv(cfg.output_dir / "tables" / "illustrative_hvac_scenarios.csv", index=False)
print("NOTE: illustrative user-specified scenarios only. Scenario labels are generic; "
      "no model-specific savings are claimed, and H5 remains unsupported.")


## 13. Unseen-building and unseen-site generalization (evaluation tasks 2 and 3)

The evaluation protocol defines three tasks (Section *Leakage-Aware Evaluation Tasks*). Task 1 (same-building chronological forecasting) is executed above. This section executes the remaining two:

- **Unseen-building generalization** — buildings (not rows) are assigned to mutually exclusive train/validation/test groups (70/15/15 at the *building* level, `GroupShuffleSplit`, seed 42); imputer and scaler are refit on the train buildings only.
- **Unseen-site generalization** — leave-one-site-out across every site represented in the sample; within each fold the validation split is drawn from *train-site* buildings only, so the held-out site is never touched before testing.

Hyperparameter grids, the selection rule, and seeds are identical to the executed run (selection on the transfer validation split only; no fold-specific tuning beyond that). The ID-free GRU is retrained per task at L=24 (5 seeds for unseen-building; 1 seed per fold for unseen-site).

Results are collected in a **separate registry** (`bc_rows` / `bc_store`) and exported to their own tables, so every table, paired test, and Holm correction of the executed same-building evaluation remains bit-identical. Leakage tripwires assert that no test building (respectively site) appears in any fold's training data.

In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers for the generalization tasks (bc_* namespace; the executed
# same-building results are untouched).
# ---------------------------------------------------------------------------
from sklearn.model_selection import GroupShuffleSplit

bc_rows = []          # metric rows: {Task, Model, **regression_metrics, Train_seconds}
bc_store = {"unseen_building": {"preds_val": {}, "preds_test": {}},
            "unseen_site": {"preds_val": {}, "preds_test": {}}}
bc_paired_rows = []
bc_timings = {}

def _group_splits_by(df, col, seed, test_size=0.15):
    """70/15/15 split at the GROUP level (groups disjoint across partitions)."""
    idx = np.arange(len(df))
    groups = df[col].to_numpy()
    trval_pos, te_pos = next(GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=seed).split(idx, groups=groups))
    tr_pos, va_pos = next(GroupShuffleSplit(
        n_splits=1, test_size=0.15 / 0.85, random_state=seed + 1
    ).split(trval_pos, groups=groups[trval_pos]))
    return {"train": idx[trval_pos[tr_pos]], "val": idx[trval_pos[va_pos]], "test": idx[te_pos]}

def _bc_fit_tree(kind, bundle, y_tr, y_va):
    """Fit one tree model with the SAME grids/hyperparameters/seed as the
    executed run; select on the transfer validation split only."""
    start = time.perf_counter()
    if kind == "lgb":
        best_model, best_val = None, np.inf
        for num_leaves in cfg.lgb_num_leaves_grid:
            for feat_frac in cfg.lgb_feature_fraction_grid:
                for min_child in cfg.lgb_min_child_samples_grid:
                    for bag_frac in cfg.lgb_bagging_fraction_grid:
                        model = _lgb_from_params({"num_leaves": num_leaves,
                                                  "feature_fraction": feat_frac,
                                                  "min_child_samples": min_child,
                                                  "bagging_fraction": bag_frac,
                                                  "bagging_freq": 1 if bag_frac < 1.0 else 0})
                        model.fit(bundle.X_train, y_tr,
                                  eval_set=[(bundle.X_val, y_va)], eval_metric="rmse",
                                  callbacks=[lgb.early_stopping(cfg.early_stopping_rounds, verbose=False)])
                        val_rmse = math.sqrt(mean_squared_error(y_va, model.predict(bundle.X_val)))
                        if val_rmse < best_val:
                            best_model, best_val = model, val_rmse
    elif kind == "xgb":
        best_model = XGBRegressor(objective="reg:squarederror", eval_metric="rmse",
                                  learning_rate=cfg.xgb_cat_learning_rate,
                                  max_depth=cfg.xgb_cat_max_depth,
                                  n_estimators=cfg.boost_rounds,
                                  early_stopping_rounds=cfg.early_stopping_rounds,
                                  tree_method="hist", random_state=cfg.random_seed,
                                  n_jobs=cfg.n_jobs)
        best_model.fit(bundle.X_train, y_tr, eval_set=[(bundle.X_val, y_va)], verbose=False)
    elif kind == "cat":
        best_model = CatBoostRegressor(loss_function="RMSE", eval_metric="RMSE",
                                       learning_rate=cfg.xgb_cat_learning_rate,
                                       depth=cfg.xgb_cat_max_depth,
                                       iterations=cfg.boost_rounds,
                                       random_seed=cfg.random_seed,
                                       thread_count=cfg.n_jobs, verbose=False,
                                       allow_writing_files=False)
        best_model.fit(bundle.X_train, y_tr, eval_set=(bundle.X_val, y_va),
                       early_stopping_rounds=cfg.early_stopping_rounds, verbose=False)
    else:
        raise ValueError(kind)
    seconds = time.perf_counter() - start
    return best_model.predict(bundle.X_val), best_model.predict(bundle.X_test), seconds

def _bc_register(task, name, val_pred, y_te, test_pred, seconds):
    m = regression_metrics(y_te, test_pred)
    bc_rows.append({"Task": task, "Model": name, **m, "Train_seconds": float(seconds)})
    bc_store[task]["preds_val"][name] = np.asarray(val_pred, dtype=np.float32)
    bc_store[task]["preds_test"][name] = np.asarray(test_pred, dtype=np.float32)
    print(f"[{task}] {name}: " + ", ".join(f"{k}={v:.5f}" for k, v in m.items()))

def _gru_transfer_eval(task, splits_dict, L=24, seeds=(42,)):
    """Retrain the ID-free GRU on an arbitrary split dict using the notebook's
    own window/training machinery: the global split table is temporarily
    rebound so build_sequences and the feature-matrix normalization are
    recomputed for the transfer split with identical logic."""
    global splits
    _backup = splits
    try:
        splits = splits_dict
        X_tr, y_tr, _ = build_sequences(df, SEQUENCE_FEATURES, L, "train",
                                        cfg.max_train_windows_per_building, cfg.random_seed)
        X_va, y_va, _ = build_sequences(df, SEQUENCE_FEATURES, L, "val",
                                        cfg.max_eval_windows_per_building, cfg.random_seed)
        X_te, y_te, pos_te = build_sequences(df, SEQUENCE_FEATURES, L, "test",
                                             cfg.max_eval_windows_per_building, cfg.random_seed)
    finally:
        splits = _backup
    if len(X_tr) == 0 or len(X_va) == 0 or len(X_te) == 0:
        raise RuntimeError(f"GRU transfer ({task}): empty window sets")
    tr_mask = np.isin(np.arange(len(df)), splits_dict["train"])
    t_mu = float(np.nanmean(df[SEQ_TARGET_COL].to_numpy(dtype=np.float32)[tr_mask]))
    t_sd = max(float(np.nanstd(df[SEQ_TARGET_COL].to_numpy(dtype=np.float32)[tr_mask])), 1e-6)

    def _to_log(p):
        return p * t_sd + t_mu

    per_seed = []
    input_dim = X_tr.shape[2]
    for seed in seeds:
        set_global_seed(seed)
        model = GRUSeqRegressor(input_dim, hidden_dim=cfg.gru_hidden)
        model, _bv, _secs = _train_one(model, X_tr, y_tr, X_va, y_va, seed)
        val_log = _to_log(predict_torch(model, X_va))
        val_rmse = math.sqrt(mean_squared_error(_to_log(y_va), val_log))
        test_log = _to_log(predict_torch(model, X_te))
        per_seed.append({"seed": int(seed), "val_rmse_log": float(val_rmse),
                         "metrics": regression_metrics(_to_log(y_te), test_log),
                         "val_pred_log": val_log, "test_pred_log": test_log,
                         "seconds": float(_secs)})
        print(f"[{task}] GRU L={L} seed={seed}: val RMSE(log)={val_rmse:.5f} "
              f"test RMSE(log)={per_seed[-1]['metrics']['RMSE_log']:.5f}")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    y_test_log = _to_log(y_te)
    n_test_windows = int(len(y_te))
    del X_tr, y_tr, X_va, y_va, X_te, y_te
    gc.collect()
    test_sorted = np.sort(np.asarray(splits_dict["test"]))
    ranks = np.searchsorted(test_sorted, pos_te)
    assert np.array_equal(test_sorted[ranks], pos_te), "GRU window rows misaligned with split rows"
    mask = np.zeros(len(splits_dict["test"]), dtype=bool)
    mask[ranks] = True
    best = min(per_seed, key=lambda r: r["val_rmse_log"])
    return {"L": L, "per_seed": per_seed, "best": best,
            "y_test_log": y_test_log, "mask": mask,
            "n_test_windows": n_test_windows}

def _bc_paired(task, y_true, pred_a, pred_b, units, name_a, name_b, comparison):
    """Paired squared-loss-difference test (unit = building or site) - the same
    statistic as the executed protocol's per-building test."""
    stat, p = building_level_loss_test(y_true, pred_a, pred_b, units)
    row = {"Task": task, "Comparison": comparison, "A": name_a, "B": name_b,
           "n_units": int(len(np.unique(units))), "t_stat": stat, "p_value": p}
    bc_paired_rows.append(row)
    return row

def _holm_for_task(task):
    rows = [r for r in bc_paired_rows if r["Task"] == task]
    adj = holm_correct([r["p_value"] for r in rows])
    for r, a in zip(rows, adj):
        r["p_holm"] = float(a)

def _bc_free_matrices(mats):
    for _b in mats.values():
        _b.X_train = None
        _b.X_val = None
        _b.X_test = None
    del mats
    gc.collect()


In [ ]:
# ---------------------------------------------------------------------------
# Task 2: unseen-building generalization.
# ---------------------------------------------------------------------------
_t0 = time.perf_counter()
splits_b = _group_splits_by(df, "building_id", cfg.random_seed)
_b_counts = {k: int(df.iloc[v]["building_id"].nunique()) for k, v in splits_b.items()}
print("unseen-building split - buildings:", _b_counts,
      "| rows:", {k: int(len(v)) for k, v in splits_b.items()})

_b_tr = set(df.iloc[splits_b["train"]]["building_id"].unique().tolist())
_b_va = set(df.iloc[splits_b["val"]]["building_id"].unique().tolist())
_b_te = set(df.iloc[splits_b["test"]]["building_id"].unique().tolist())
assert _b_te.isdisjoint(_b_tr) and _b_te.isdisjoint(_b_va) and _b_tr.isdisjoint(_b_va), \
    "unseen-building leakage: a building appears in more than one partition"
print("[tripwire] unseen-building partitions are building-disjoint: OK")

y_tr_b = df.iloc[splits_b["train"]]["target_log"].to_numpy(dtype=np.float32)
y_va_b = df.iloc[splits_b["val"]]["target_log"].to_numpy(dtype=np.float32)
y_te_b = df.iloc[splits_b["test"]]["target_log"].to_numpy(dtype=np.float32)
b_ids_b = df.iloc[splits_b["test"]]["building_id"].to_numpy()
s_ids_b = df.iloc[splits_b["test"]]["site_id"].to_numpy()

matrices_b = {k: prepare_feature_set(df, feature_sets[k], splits_b)
              for k in ("base", "lag", "enthalpy_lag")}

for _lag in (1, 24, 168):
    _col = f"meter_log_lag_{_lag}"
    _bc_register("unseen_building", f"Persistence (lag-{_lag})",
                 df.iloc[splits_b["val"]][_col].to_numpy(dtype=np.float32),
                 y_te_b,
                 df.iloc[splits_b["test"]][_col].to_numpy(dtype=np.float32), 0.0)

_vp, _tp, _s = _bc_fit_tree("lgb", matrices_b["base"], y_tr_b, y_va_b)
_bc_register("unseen_building", "LightGBM (No Enthalpy)", _vp, y_te_b, _tp, _s)
_vp, _tp, _s = _bc_fit_tree("lgb", matrices_b["lag"], y_tr_b, y_va_b)
_bc_register("unseen_building", "LightGBM + Lag Features", _vp, y_te_b, _tp, _s)
_vp, _tp, _s = _bc_fit_tree("lgb", matrices_b["enthalpy_lag"], y_tr_b, y_va_b)
_bc_register("unseen_building", "LightGBM + Enthalpy + Lag", _vp, y_te_b, _tp, _s)
_vp, _tp, _s = _bc_fit_tree("xgb", matrices_b["lag"], y_tr_b, y_va_b)
_bc_register("unseen_building", "XGBoost + Lag Features", _vp, y_te_b, _tp, _s)
_vp, _tp, _s = _bc_fit_tree("cat", matrices_b["lag"], y_tr_b, y_va_b)
_bc_register("unseen_building", "CatBoost + Lag Features", _vp, y_te_b, _tp, _s)

_gru_b = _gru_transfer_eval("unseen_building", splits_b, L=24,
                            seeds=list(range(cfg.random_seed, cfg.random_seed + cfg.n_seeds)))
_GRU_B_NAME = f"GRU sequence 24h (best of {cfg.n_seeds} seeds)"
_bc_register("unseen_building", _GRU_B_NAME,
             _gru_b["best"]["val_pred_log"], _gru_b["y_test_log"],
             _gru_b["best"]["test_pred_log"], _gru_b["best"]["seconds"])

for _a, _b in (("LightGBM + Lag Features", "Persistence (lag-1)"),
               ("LightGBM + Lag Features", "XGBoost + Lag Features"),
               ("LightGBM + Lag Features", "CatBoost + Lag Features"),
               ("LightGBM + Lag Features", "LightGBM (No Enthalpy)")):
    _bc_paired("unseen_building", y_te_b,
               bc_store["unseen_building"]["preds_test"][_a],
               bc_store["unseen_building"]["preds_test"][_b],
               b_ids_b, _a, _b, f"{_a} vs {_b} [unseen-building]")
_m = _gru_b["mask"]
_bc_paired("unseen_building", _gru_b["y_test_log"],
           bc_store["unseen_building"]["preds_test"]["LightGBM + Lag Features"][_m],
           _gru_b["best"]["test_pred_log"],
           b_ids_b[_m], "LightGBM + Lag Features", _GRU_B_NAME,
           f"LightGBM + Lag Features vs GRU (matched rows, n={int(_m.sum())}) [unseen-building]")
_holm_for_task("unseen_building")

ub_ci_rows = []
for _name, _pred in bc_store["unseen_building"]["preds_test"].items():
    if len(_pred) != len(y_te_b):
        continue  # matched-window models (GRU) get their own CI below
    low, high = bootstrap_rmse_by_building(y_te_b, _pred, b_ids_b,
                                           iterations=cfg.bootstrap_iterations,
                                           seed=cfg.random_seed)
    ub_ci_rows.append({"Task": "unseen_building", "Model": _name,
                       "RMSE_log": math.sqrt(mean_squared_error(y_te_b, _pred)),
                       "CI_95_low": low, "CI_95_high": high, "Coverage": "full test rows"})
_low, _high = bootstrap_rmse_by_building(_gru_b["y_test_log"], _gru_b["best"]["test_pred_log"],
                                         b_ids_b[_m], iterations=cfg.bootstrap_iterations,
                                         seed=cfg.random_seed)
ub_ci_rows.append({"Task": "unseen_building", "Model": _GRU_B_NAME,
                   "RMSE_log": math.sqrt(mean_squared_error(_gru_b["y_test_log"],
                                                            _gru_b["best"]["test_pred_log"])),
                   "CI_95_low": _low, "CI_95_high": _high,
                   "Coverage": f"matched rows only (n={int(_m.sum())})"})

bc_store["unseen_building"].update({
    "y_true": y_te_b, "units": b_ids_b, "sites": s_ids_b,
    "gru_matched_mask": _m, "gru_y_test_log": _gru_b["y_test_log"],
    "gru_per_seed": [{k: v for k, v in r.items()
                      if k not in ("pred_log", "val_pred_log", "test_pred_log")}
                     for r in _gru_b["per_seed"]],
    "n_train_val_test_buildings": _b_counts,
})
_bc_free_matrices(matrices_b)
bc_timings["unseen_building_seconds"] = time.perf_counter() - _t0
mem_report("after unseen-building task")


In [ ]:
# ---------------------------------------------------------------------------
# Task 3: unseen-site generalization (leave-one-site-out). Within each fold the
# validation split is drawn from TRAIN-SITE buildings only; 1 GRU seed per fold.
# Wrapped in try/except so an unexpected LOSO failure cannot cost the
# unseen-building exports (the run reports the failure loudly instead).
# ---------------------------------------------------------------------------
_t0 = time.perf_counter()
loso_ok = False
loso_per_site, loso_units, loso_y, _gru_folds = [], {}, {}, []
try:
    _sites = sorted(int(s) for s in df["site_id"].unique())
    print(f"leave-one-site-out over {len(_sites)} sites: {_sites}")

    for _s in _sites:
        _te = np.flatnonzero(df["site_id"].to_numpy() == _s)
        _trv = np.flatnonzero(df["site_id"].to_numpy() != _s)
        _tr_pos, _va_pos = next(GroupShuffleSplit(
            n_splits=1, test_size=0.15, random_state=cfg.random_seed
        ).split(_trv, groups=df.iloc[_trv]["building_id"].to_numpy()))
        splits_s = {"train": _trv[_tr_pos], "val": _trv[_va_pos], "test": _te}

        assert _s not in set(df.iloc[splits_s["train"]]["site_id"].unique().tolist()), \
            f"LOSO leakage: site {_s} present in its own training fold"
        assert set(df.iloc[_te]["building_id"].unique().tolist()).isdisjoint(
            set(df.iloc[splits_s["train"]]["building_id"].unique().tolist())), \
            f"LOSO leakage at site {_s}: test buildings appear in the train fold"

        y_tr_s = df.iloc[splits_s["train"]]["target_log"].to_numpy(dtype=np.float32)
        y_va_s = df.iloc[splits_s["val"]]["target_log"].to_numpy(dtype=np.float32)
        y_te_s = df.iloc[splits_s["test"]]["target_log"].to_numpy(dtype=np.float32)

        matrices_s = {k: prepare_feature_set(df, feature_sets[k], splits_s)
                      for k in ("base", "lag", "enthalpy_lag")}

        def _reg(name, val_pred, test_pred, seconds):
            m = regression_metrics(y_te_s, test_pred)
            loso_per_site.append({"Site": _s, "Model": name, **m})
            loso_units.setdefault(name, {})[int(_s)] = np.asarray(test_pred, dtype=np.float32)
            loso_y[int(_s)] = y_te_s

        _reg("Persistence (lag-1)",
             df.iloc[splits_s["val"]]["meter_log_lag_1"].to_numpy(dtype=np.float32),
             df.iloc[splits_s["test"]]["meter_log_lag_1"].to_numpy(dtype=np.float32), 0.0)
        _vp, _tp, _s2 = _bc_fit_tree("lgb", matrices_s["base"], y_tr_s, y_va_s)
        _reg("LightGBM (No Enthalpy)", _vp, _tp, _s2)
        _vp, _tp, _s2 = _bc_fit_tree("lgb", matrices_s["lag"], y_tr_s, y_va_s)
        _reg("LightGBM + Lag Features", _vp, _tp, _s2)
        _vp, _tp, _s2 = _bc_fit_tree("lgb", matrices_s["enthalpy_lag"], y_tr_s, y_va_s)
        _reg("LightGBM + Enthalpy + Lag", _vp, _tp, _s2)
        _vp, _tp, _s2 = _bc_fit_tree("xgb", matrices_s["lag"], y_tr_s, y_va_s)
        _reg("XGBoost + Lag Features", _vp, _tp, _s2)
        _vp, _tp, _s2 = _bc_fit_tree("cat", matrices_s["lag"], y_tr_s, y_va_s)
        _reg("CatBoost + Lag Features", _vp, _tp, _s2)

        _gru_s = _gru_transfer_eval("unseen_site", splits_s, L=24, seeds=(cfg.random_seed,))
        _GRU_S_NAME = "GRU sequence 24h (1 seed)"
        _gru_folds.append({
            "site": int(_s),
            "y_log": _gru_s["y_test_log"],
            "gru_pred": _gru_s["best"]["test_pred_log"],
            "lgb_pred_on_windows":
                loso_units["LightGBM + Lag Features"][int(_s)][_gru_s["mask"]],
            "n_windows": int(_gru_s["n_test_windows"]),
            "metrics": _gru_s["best"]["metrics"],
            "seconds": _gru_s["best"]["seconds"],
        })

        _bc_free_matrices(matrices_s)

    # pooled LOSO metrics: every test row appears in exactly one held-out fold
    _sites_sorted = sorted(loso_y)
    us_y = np.concatenate([loso_y[s] for s in _sites_sorted])
    us_row_sites = np.concatenate([np.full(len(loso_y[s]), s, dtype=np.int64) for s in _sites_sorted])
    us_building_ids = np.concatenate([
        df.iloc[np.flatnonzero(df["site_id"].to_numpy() == s)]["building_id"].to_numpy()
        for s in _sites_sorted])
    loso_rows = []
    for _name in loso_units:
        _p = np.concatenate([loso_units[_name][s] for s in _sites_sorted])
        loso_rows.append({"Task": "unseen_site", "Model": _name,
                          **regression_metrics(us_y, _p), "Train_seconds": float("nan")})
        bc_store["unseen_site"]["preds_test"][_name] = _p

    # GRU pooled over its own matched windows across folds (NOT concatenated with
    # the full-coverage tree predictions).
    _GRU_S_NAME = "GRU sequence 24h (1 seed)"
    _gru_y = np.concatenate([f["y_log"] for f in _gru_folds])
    _gru_p = np.concatenate([f["gru_pred"] for f in _gru_folds])
    _gru_lgb = np.concatenate([f["lgb_pred_on_windows"] for f in _gru_folds])
    _gru_units = np.concatenate([np.full(f["n_windows"], f["site"], dtype=np.int64)
                                 for f in _gru_folds])
    loso_rows.append({"Task": "unseen_site", "Model": _GRU_S_NAME,
                      **regression_metrics(_gru_y, _gru_p),
                      "Train_seconds": float(sum(f["seconds"] for f in _gru_folds))})
    bc_store["unseen_site"].update({
        "gru": {"y_log": _gru_y, "pred_log": _gru_p, "lgb_pred_on_windows": _gru_lgb,
                "window_sites": _gru_units},
        "gru_per_fold": [{k: f[k] for k in ("site", "n_windows", "metrics", "seconds")}
                         for f in _gru_folds],
    })
    bc_rows.extend(loso_rows)
    bc_store["unseen_site"].update({"y_true": us_y, "row_sites": us_row_sites,
                                    "building_ids": us_building_ids, "fold_sites": _sites_sorted})

    for _a, _b in (("LightGBM + Lag Features", "Persistence (lag-1)"),
                   ("LightGBM + Lag Features", "XGBoost + Lag Features"),
                   ("LightGBM + Lag Features", "CatBoost + Lag Features"),
                   ("LightGBM + Lag Features", "LightGBM (No Enthalpy)")):
        _bc_paired("unseen_site", us_y,
                   bc_store["unseen_site"]["preds_test"][_a],
                   bc_store["unseen_site"]["preds_test"][_b],
                   us_row_sites, _a, _b, f"{_a} vs {_b} [unseen-site]")
    _bc_paired("unseen_site", _gru_y, _gru_lgb, _gru_p, _gru_units,
               "LightGBM + Lag Features", _GRU_S_NAME,
               f"LightGBM + Lag Features vs GRU (matched windows, n={len(_gru_y)}) [unseen-site]")
    _holm_for_task("unseen_site")
    loso_ok = True

except Exception:
    import traceback
    print("[unseen_site] FAILED - unseen-building exports are unaffected; unseen-site exports will be skipped.")
    traceback.print_exc()
bc_timings["unseen_site_seconds"] = time.perf_counter() - _t0
mem_report("after unseen-site task")


In [ ]:
# ---------------------------------------------------------------------------
# Export the generalization results (separate files from the executed run's
# tables, which remain untouched).
# ---------------------------------------------------------------------------
import re as _re

ub_df = pd.DataFrame([r for r in bc_rows if r["Task"] == "unseen_building"])
us_pooled_df = pd.DataFrame([r for r in bc_rows if r["Task"] == "unseen_site"])
us_folds_df = pd.DataFrame(loso_per_site)
paired_df = pd.DataFrame(bc_paired_rows)
ci_df = pd.DataFrame(ub_ci_rows)

ub_df.to_csv(cfg.output_dir / "tables" / "unseen_building_results.csv", index=False)
if loso_ok:
    us_folds_df.to_csv(cfg.output_dir / "tables" / "unseen_site_results_per_fold.csv", index=False)
    us_pooled_df.to_csv(cfg.output_dir / "tables" / "unseen_site_results_pooled.csv", index=False)
else:
    print("[warn] unseen-site CSV exports skipped (LOSO did not complete)")
paired_df.to_csv(cfg.output_dir / "tables" / "unseen_generalization_paired_tests.csv", index=False)
ci_df.to_csv(cfg.output_dir / "tables" / "unseen_building_bootstrap_ci.csv", index=False)

def _san(name):
    return _re.sub(r"\W+", "_", name)

npz_args = {}
for _name, _p in bc_store["unseen_building"]["preds_test"].items():
    npz_args["ub_pred__" + _san(_name)] = _p
npz_args.update({
    "ub_y_true": bc_store["unseen_building"]["y_true"],
    "ub_building_id": bc_store["unseen_building"]["units"],
    "ub_site_id": bc_store["unseen_building"]["sites"],
    "ub_gru_matched_mask": bc_store["unseen_building"]["gru_matched_mask"],
    "ub_gru_y_test_log": bc_store["unseen_building"]["gru_y_test_log"],
})
if loso_ok:
    npz_args.update({
        "us_y_true": bc_store["unseen_site"]["y_true"],
        "us_row_site": bc_store["unseen_site"]["row_sites"],
        "us_building_id": bc_store["unseen_site"]["building_ids"],
        "us_gru_y_log": bc_store["unseen_site"]["gru"]["y_log"],
        "us_gru_pred_log": bc_store["unseen_site"]["gru"]["pred_log"],
        "us_gru_lgb_pred_log": bc_store["unseen_site"]["gru"]["lgb_pred_on_windows"],
        "us_gru_window_site": bc_store["unseen_site"]["gru"]["window_sites"],
    })
    for _name, _p in bc_store["unseen_site"]["preds_test"].items():
        npz_args["us_pred__" + _san(_name)] = _p
else:
    print("[warn] unseen-site npz entries skipped (LOSO did not complete)")
np.savez_compressed(cfg.output_dir / "tables" / "bc_predictions.npz", **npz_args)

bc_summary = {
    "build": TROPILITE_BUILD,
    "tasks": {
        "unseen_building": {
            "split": "70/15/15 at the building level (GroupShuffleSplit, seed 42)",
            "buildings_per_partition": bc_store["unseen_building"]["n_train_val_test_buildings"],
            "leakage_tripwires": "building-disjoint partitions asserted",
            "gru": {"L": 24, "seeds": cfg.n_seeds,
                    "per_seed": bc_store["unseen_building"]["gru_per_seed"]},
        },
    },
    "timings_seconds": bc_timings,
    "pooled_unseen_site_metrics": us_pooled_df.to_dict(orient="records"),
}
if loso_ok:
    bc_summary["tasks"]["unseen_site"] = {
        "split": "leave-one-site-out; per-fold validation drawn from train-site buildings only",
        "sites": bc_store["unseen_site"]["fold_sites"],
        "gru": {"L": 24, "seeds": 1,
                "per_fold": bc_store["unseen_site"]["gru_per_fold"]},
    }
else:
    bc_summary["tasks"]["unseen_site"] = {"status": "not completed in this run"}
def _json_default(o):
    if hasattr(o, "tolist"):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, float) and o != o:
        return None
    return str(o)

with open(cfg.output_dir / "unseen_generalization_summary.json", "w", encoding="utf-8") as _f:
    json.dump(bc_summary, _f, indent=2, default=_json_default)

print("unseen-building results:")
print(ub_df[["Model", "RMSE_log", "R2_log", "MAE_log"]].to_string(index=False))
if loso_ok:
    print("\nunseen-site pooled results (all rows held out exactly once):")
    print(us_pooled_df[["Model", "RMSE_log", "R2_log", "MAE_log"]].to_string(index=False))
print("\npaired tests:")
print(paired_df[["Task", "Comparison", "n_units", "t_stat", "p_value", "p_holm"]].to_string(index=False))
print("\nGeneralization tables written to:", cfg.output_dir / "tables")


## 16. Final experiment manifest


In [ ]:
manifest = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()},
    "data": {
        "rows": len(df),
        "buildings": int(df["building_id"].nunique()),
        "sites": int(df["site_id"].nunique()),
        "tropical_rows": int(df["is_tropical"].sum()),
        "split_sizes": {k: int(len(v)) for k, v in splits.items()},
    },
    "final_ensemble": {"name": FINAL_ENSEMBLE_NAME, "alpha_gru": FINAL_ALPHA, "alpha_lgbm": 1-FINAL_ALPHA},
    "versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "lightgbm": lgb.__version__,
        "torch": torch.__version__,
    },
}
with open(cfg.output_dir / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Outputs written to:", cfg.output_dir.resolve())
print("Final model:", FINAL_ENSEMBLE_NAME)
_final_pred = predictions_test[_FULL_NAME]
if len(_final_pred) != len(y_test):
    _final_metrics = regression_metrics(SEQ_TEST["y_log"], _final_pred)
else:
    _final_metrics = regression_metrics(y_test, _final_pred)
print("Final global metrics:", _final_metrics)


## Recommended paper reporting workflow

Use the notebook-generated CSV tables rather than typing values manually. For the manuscript, report:

- exact data acquisition date, the electricity-only filter, and the building-level sample;
- the per-building chronological split (70/15/15) with the leakage assertions passing;
- software versions from `experiment_manifest.json`;
- global results on both log and raw-kWh scales with building-level bootstrap CIs;
- the validation-only fusion search and whether the retention gate passed (alpha, improvement, seed consistency);
- all-seed GRU test results (`gru_test_all_seeds.csv`), mean +- sd;
- pre-registered primary comparisons with Holm correction; all other tests labelled exploratory;
- the H2 tropical-subset decision from `h2_tropical_decision.csv`;
- the zero-inflation diagnostic;
- local latency as local-machine measurements only; H4 (Raspberry Pi) remains unvalidated;
- HVAC savings only as the labelled illustrative scenarios; H5 remains unsupported.

## Final Protocol A reporting checklist

Export:
- climate verification table;
- chronological split manifest;
- leakage test results;
- ablation tables;
- both log and inverse-scale metrics;
- Holm-corrected comparisons;
- bootstrap confidence intervals;
- building-level statistics;
- final SHAP analysis;
- zero-inflation diagnostics;
- configuration and seed records.

In [ ]:

# ============================================================
# FINAL PAPER-READY EXPERIMENT EXPORT
# This cell only reads existing notebook objects and exports them.
# It does not modify training, evaluation, or experiment results.
# ============================================================

from pathlib import Path
import json, zipfile, shutil, csv, inspect, datetime, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPORT_ROOT = Path("final_results")
JSON_DIR = EXPORT_ROOT / "json"
FIG_DIR = EXPORT_ROOT / "figures"
CSV_DIR = EXPORT_ROOT / "csv"

for d in [JSON_DIR, FIG_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def make_serializable(obj):
    """Convert notebook objects into JSON-safe structures."""
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(x) for x in obj]
    try:
        return str(obj)
    except Exception:
        return None

def save_json(data, filename):
    with open(JSON_DIR / filename, "w", encoding="utf-8") as f:
        json.dump(make_serializable(data), f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# Collect existing experiment results
# ------------------------------------------------------------
experiment_results = {}

existing_candidates = [
    "results",
    "results_df",
    "manifest",
    "metrics",
    "model_comparison",
    "gru_seed_df",
    "benchmark_results",
    "latency_results",
    "shap_importance_df",
]

for name in existing_candidates:
    if name in globals():
        try:
            experiment_results[name] = globals()[name]
        except Exception:
            pass

# Explicit model comparison export
if "results_df" in globals():
    try:
        results_df.to_csv(CSV_DIR / "metrics.csv", index=False)
    except Exception:
        pass
elif "results" in globals():
    try:
        pd.DataFrame(results).to_csv(CSV_DIR / "metrics.csv", index=False)
    except Exception:
        pass

# Individual model results when result registry exists
individual_results = {}
if "results" in globals():
    try:
        if isinstance(results, list):
            for row in results:
                if isinstance(row, dict):
                    name = row.get("Model", row.get("model", f"model_{len(individual_results)}"))
                    individual_results[str(name)] = row
    except Exception:
        pass

save_json(individual_results, "experiment_results.json")

# Model comparison
comparison = {}
if "results_df" in globals():
    comparison = results_df.to_dict(orient="records")
elif "results" in globals():
    comparison = results
save_json(comparison, "model_comparison.json")

# Full raw collected objects
save_json(experiment_results, "all_collected_results.json")

# ------------------------------------------------------------
# Build final summary
# ------------------------------------------------------------
summary = {}

for key in [
    "manifest",
    "cfg",
    "feature_sets",
    "FINAL_ENSEMBLE_NAME",
    "FINAL_ALPHA",
    "BEST_L",
    "best_L",
    "TREE_NAME",
    "GRU_NAME",
]:
    if key in globals():
        summary[key] = globals()[key]

if "results_df" in globals():
    try:
        summary["final_test_metrics"] = results_df.to_dict(orient="records")
    except Exception:
        pass

if "df" in globals():
    try:
        summary["dataset_information"] = {
            "rows": len(df),
            "columns": list(df.columns),
        }
    except Exception:
        pass

summary["export_time"] = datetime.datetime.utcnow().isoformat()
save_json(summary, "summary.json")

# ------------------------------------------------------------
# Export currently created matplotlib figures
# ------------------------------------------------------------
figure_files = []

def export_figure(fig, base):
    try:
        svg = FIG_DIR / f"{base}.svg"
        pdf = FIG_DIR / f"{base}.pdf"
        fig.savefig(svg, bbox_inches="tight")
        fig.savefig(pdf, bbox_inches="tight")
        figure_files.extend([str(svg), str(pdf)])
    except Exception as e:
        print("Figure export skipped:", base, e)

# Save all active figures
try:
    figs = [plt.figure(n) for n in plt.get_fignums()]
    for i, fig in enumerate(figs, 1):
        export_figure(fig, f"figure_{i}")
except Exception:
    pass

# Copy already generated important figures from output directory
possible_dirs = []
if "cfg" in globals():
    try:
        possible_dirs.append(Path(cfg.output_dir))
    except Exception:
        pass
possible_dirs.append(Path("outputs"))

copied = set()
for base_dir in possible_dirs:
    if base_dir.exists():
        for p in base_dir.rglob("*"):
            if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".svg", ".pdf"]:
                if p.resolve() not in copied:
                    try:
                        target = FIG_DIR / p.name
                        if not target.exists():
                            shutil.copy2(p, target)
                        copied.add(p.resolve())
                        figure_files.append(str(target))
                    except Exception:
                        pass

# ------------------------------------------------------------
# README
# ------------------------------------------------------------
readme = """TropiLite-IE Final Results Export

Folders:
- json/: Structured experiment results and summaries.
- figures/: Paper-ready exported figures (SVG/PDF preferred).
- csv/: Tabular metrics for analysis and manuscript preparation.

Files:
- summary.json: experiment configuration, dataset information, final metrics.
- experiment_results.json: individual experiment/model results.
- model_comparison.json: comparison table between models.
- metrics.csv: model metrics table when available.

This archive was generated automatically from existing notebook variables.
No training or evaluation code was modified.
"""

(EXPORT_ROOT / "README.txt").write_text(readme, encoding="utf-8")

# ------------------------------------------------------------
# Create ZIP archive
# ------------------------------------------------------------
zip_path = Path("final_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in EXPORT_ROOT.rglob("*"):
        if file.is_file():
            z.write(file, file.as_posix())

# Verification
zip_ok = zip_path.exists()
zip_files = []
if zip_ok:
    with zipfile.ZipFile(zip_path, "r") as z:
        zip_files = z.namelist()

print("=" * 60)
print("FINAL EXPORT COMPLETE")
print("Directory:", EXPORT_ROOT.resolve())
print("ZIP:", zip_path.resolve())
print("ZIP created:", zip_ok)
print("Files in ZIP:", len(zip_files))
for f in zip_files[:20]:
    print(" -", f)
if len(zip_files) > 20:
    print(" ...")
print("=" * 60)
